In [ ]:
# === FORCE-REINSTALL torch first (wipes all packages installed before it) ===
!pip install -q "numpy<2.0" || echo "WARN: numpy downgrade failed"
!pip install -q "huggingface_hub>=0.27.0" "kagglehub>=0.3.0" || echo "WARN: hub install failed"
!pip install -q transformers>=4.49.0 || echo "WARN: transformers install failed"

# FORCE-REINSTALL torch for P100/T4/A10G (sm_60/sm_75/sm_89)
!pip install torch==2.5.1+cu121 torchvision torchaudio --force-reinstall --quiet --index-url https://download.pytorch.org/whl/cu121 || echo "WARN: torch reinstall failed"
!pip install -q "numpy<2.0" --force-reinstall --no-deps || echo "WARN: numpy final downgrade failed"
!pip install -q --upgrade "scipy>=1.14.1" || echo "WARN: scipy upgrade failed"

# === ALL packages that must survive: install AFTER torch force-reinstall ===
!pip install -q fastapi uvicorn python-multipart "pillow==10.4.0" "trimesh<4.0" gdown opencv-python pyyaml packaging || echo "WARN: core pip install had failures"
!pip install -q git+https://github.com/facebookresearch/segment-anything-2.git || echo "WARN: SAM2 install failed"
!pip install onnxruntime openmesh python-multipart 2>&1 | tail -5 || echo "WARN: onnxruntime install failed"
!pip install -q rembg --no-deps 2>&1 | tail -3 || echo "WARN: rembg install failed"
!pip install -q onnxruntime Pillow numpy scipy pooch 2>&1 | tail -3 || echo "WARN: rembg deps install failed"
!pip install -q torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.5.1+cu121.html || echo "WARN: torch-scatter install failed (non-critical)"

# pytorch3d: try build from source - falls back to trimesh shim
!pip install -q nvidia-cuda-nvcc-cu12 nvidia-cuda-nvrtc-cu12 2>&1 | tail -3 || echo "WARN: CUDA pip toolkit failed"
!FORCE_CUDA=1 MAX_JOBS=4 pip install -q "pytorch3d @ git+https://github.com/facebookresearch/pytorch3d.git" 2>&1 | tail -5 || echo "pytorch3d build failed - trimesh shim will handle it"

# Additional deps: GarmentRec (pymeshlab, loguru) + GarmentGPT (llamafactory, fire, vector-quantize-pytorch)
!apt-get install -y -qq libgl1 libglib2.0-0 libsm6 libxext6 libxrender-dev libopengl0 2>/dev/null || true
!pip install pymeshlab loguru scikit-learn torch_geometric 2>&1 | tail -5 || echo "WARN: pymeshlab/torch_geometric install failed"
!pip install -q llamafactory fire vector-quantize-pytorch || echo "WARN: llamafactory/fire/vector-quantize install failed"
!pip install -q bitsandbytes || echo "WARN: bitsandbytes install failed"
!pip install -q nest-asyncio pymatting || echo "WARN: nest-asyncio/pymatting install failed"

print("Cell 0 complete.")

# NumPy 2.x compat shim (onnxruntime needs _blas_supports_fpe)
import numpy as _np
if not hasattr(_np._core._multiarray_umath, '_blas_supports_fpe'):
    _np._core._multiarray_umath._blas_supports_fpe = lambda *a, **k: True

# Verify critical packages
try:
    import fastapi, uvicorn, PIL, trimesh, cv2, pymeshlab, sam2, vector_quantize_pytorch
    print("All critical packages: OK")
except ImportError as e:
    print(f"WARNING: import failed: {e}")

print("Done. Continuing in same kernel...")

# Clear namespace
get_ipython().run_line_magic("reset", "-f")


In [ ]:
import os
import sys
import shutil
import zipfile
import numpy as np
import pickle
import glob
import subprocess
import time
import requests
from pathlib import Path
from huggingface_hub import hf_hub_download, login

# Disable Xet Storage native protocol (cold cache hangs on first access)
# Forces HTTP fallback via presigned URLs instead of hf_xet Rust downloader
os.environ["HF_HUB_DISABLE_XET"] = "1"

# Set HF token early (before any hf_hub_download calls for gated models)
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "hf_YOUR_TOKEN_HERE")
login(token=os.environ["HF_TOKEN"])

WD = Path("/root")
WEIGHTS_DIR = WD / "weights"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)


# ---- Robust Download Helper (multi-layer fallback, auth headers) ----
def robust_download(url, dest, expected_mb=None, max_retries=3, desc="", extra_headers=None):
    """Download with wget (resume+progress) -> requests streaming fallback.
    extra_headers: dict of extra HTTP headers (e.g., {"Authorization": "Bearer ..."})
    """
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    extra_headers = extra_headers or {}
    for attempt in range(max_retries):
        try:
            if desc:
                print(f"  {desc}: attempt {attempt+1}/{max_retries}...")
            # Layer 1: wget with resume, progress, retries, auth headers
            cmd = [
                "wget", "-c", "--show-progress",
                "--dns-timeout=15", "--connect-timeout=30", "--read-timeout=120",
                "--tries=3", "--retry-connrefused",
            ]
            for k, v in extra_headers.items():
                cmd += ["--header", f"{k}: {v}"]
            cmd += ["-O", str(dest), url]
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=7200)
            if r.returncode == 0 and dest.exists() and dest.stat().st_size > 0:
                size_mb = dest.stat().st_size / (1024*1024)
                if expected_mb is None or size_mb >= expected_mb * 0.9:
                    if desc: print(f"  {desc}: {size_mb:.0f} MB")
                    return True
            # Layer 2: requests streaming (bypasses wget issues, supports auth)
            if desc: print(f"  {desc}: wget failed, trying requests...")
            sess_headers = dict(extra_headers)
            with requests.get(url, headers=sess_headers, stream=True, timeout=300) as resp:
                resp.raise_for_status()
                with open(dest, 'wb') as f:
                    for chunk in resp.iter_content(chunk_size=8192):
                        if chunk: f.write(chunk)
            if dest.exists() and dest.stat().st_size > 0:
                size_mb = dest.stat().st_size / (1024*1024)
                if expected_mb is None or size_mb >= expected_mb * 0.9:
                    if desc: print(f"  {desc}: {size_mb:.0f} MB")
                    return True
        except Exception as e:
            if desc: print(f"  {desc}: attempt {attempt+1} failed: {e}")
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
    if desc: print(f"  *** {desc}: FAILED after {max_retries} attempts ***")
    return False


# ---- Kaggle Dataset Mirror (GCP-local, no Xet, instant) ----
KAGGLE_DATASET = "jacobthankgod/korra-garment-weights"
kaggle_path = None
try:
    import kagglehub
    kp = kagglehub.dataset_download(KAGGLE_DATASET)
    if kp and Path(kp).exists():
        kaggle_path = kp
        print(f"Kaggle Dataset '{KAGGLE_DATASET}' available: {kp}")
    else:
        print(f"Kaggle Dataset '{KAGGLE_DATASET}' not found. "
              f"Upload weights first: python scripts/upload_garment_weights_to_kaggle.py")
except Exception as e:
    print(f"Kaggle Dataset not available: {e}")


# ---- cloudflared binary (must persist after kernel restart) ----
cloudflared_bin = WD / "cloudflared"
# Must be a valid ELF binary >= 1MB (corrupt if DNS failed during download)
if not cloudflared_bin.exists() or cloudflared_bin.stat().st_size < 1_000_000:
    if cloudflared_bin.exists():
        cloudflared_bin.unlink()
        print("cloudflared: stale file removed (size < 1MB)")
    print("Downloading cloudflared...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {cloudflared_bin}
    !chmod +x {cloudflared_bin}
    import os as _os; size_kb = _os.path.getsize(str(cloudflared_bin)) / 1024
    print(f"cloudflared: done ({size_kb:.0f} KB)")
else:
    size_kb = cloudflared_bin.stat().st_size / 1024
    print(f"cloudflared: cached ({size_kb:.0f} KB)")

# ---- GarmentRec: Clone repo + download assets + model weights ----
GARMENT_REC_DIR = WD / "GarmentRec"
if not (GARMENT_REC_DIR / "code").exists():
    print("Cloning GarmentRec repo...")
    !git clone --depth=1 https://github.com/worryDes/GarmentRec.git {GARMENT_REC_DIR} 2>&1 || echo "WARN: GarmentRec clone failed (may already exist)"
else:
    print("GarmentRec repo: cached")

# ---- GarmentRec: Robust Asset Extraction ----
GARMENT_REC_DIR = WD / "GarmentRec"
_data_dir = GARMENT_REC_DIR / "data"
_tmps = _data_dir / "tmps"
midpairs_path = _data_dir / "midpairs.pkl"

# Integrity check: pickles + tmps folder (tmps from 7z contains PCA/UV for template gen)
dense_midpairs_path_c4 = _data_dir / "dense_midpairs.pkl"
assets_valid = (
    midpairs_path.exists() and midpairs_path.stat().st_size > 100
    and dense_midpairs_path_c4.exists() and dense_midpairs_path_c4.stat().st_size > 100
    and _tmps.exists()
) 

if not assets_valid:
    print("Downloading/Extracting GarmentRec assets...")
    !apt-get install -y -qq p7zip-full > /dev/null 2>&1 || echo "WARN: p7zip install failed"
    if not Path("/tmp/garmentrec_assets.7z").exists():
        !gdown 1PTbfEMchwgHpaL3y8Gm1__sbzFLqfoYj -O /tmp/garmentrec_assets.7z --quiet
    
    # Clean temp extraction
    if os.path.exists("/tmp/garmentrec_assets_tmp"): shutil.rmtree("/tmp/garmentrec_assets_tmp")
    os.makedirs("/tmp/garmentrec_assets_tmp")
    
    !7z x /tmp/garmentrec_assets.7z -o/tmp/garmentrec_assets_tmp -y > /dev/null 2>&1
    import glob
    print("Robustly locating extracted assets...")
    found_mid = glob.glob("/tmp/garmentrec_assets_tmp/**/midpairs.pkl", recursive=True)
    if found_mid:
        data_root = Path(found_mid[0]).parent
        print(f"Detected asset root: {data_root}")
        (GARMENT_REC_DIR / "data").mkdir(parents=True, exist_ok=True)
        !cp -r {data_root}/* {GARMENT_REC_DIR}/data/
    
    # Locate data root (where midpairs.pkl is)
    found = glob.glob("/tmp/garmentrec_assets_tmp/**/midpairs.pkl", recursive=True)
    if found:
        src_root = Path(found[0]).parent
        print(f"Detected asset root: {src_root}")
        _data_dir.mkdir(parents=True, exist_ok=True)
        # Copy everything safely
        for item in os.listdir(src_root):
            s = src_root / item
            d = _data_dir / item
            if s.is_dir():
                if d.exists(): shutil.rmtree(d)
                shutil.copytree(s, d)
            else:
                shutil.copy2(s, d)
        print("Assets copied successfully.")
    else:
        print("CRITICAL: Could not find midpairs.pkl in extracted archive!")
else:
    print("GarmentRec assets: cached and verified.")


# Ensure tmps folder is in the right place (7z may extract to wrong path)
tmps_target = GARMENT_REC_DIR / "data" / "tmps"
if not tmps_target.exists():
    # Search for tmps folder anywhere under GARMENT_REC_DIR
    for candidate in [
        GARMENT_REC_DIR / "tmps",
        GARMENT_REC_DIR / "models" / "tmps",
        GARMENT_REC_DIR / "archive" / "tmps",
        GARMENT_REC_DIR / "archive" / "data" / "tmps",
        GARMENT_REC_DIR / "archive" / "data" / "data" / "tmps",
        Path("/tmp/garmentrec_assets_tmp") / "data" / "tmps",
        Path("/tmp/garmentrec_assets_tmp") / "tmps",
    ]:
        if candidate.exists():
            tmps_target.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(candidate), str(tmps_target))
            print(f"Moved tmps from {candidate} to {tmps_target}")
            break

# GarmentRec model weights — multi-layer fallback: Kaggle Dataset → wget → requests
model_dir = GARMENT_REC_DIR / "models" / "mrf_0.1_shading_0.1"
model_pth = model_dir / "mrf_0.1_shading_0.1_pca64_ep100_bth0.pth"

# Remove stale/broken files (anything under 100MB is invalid)
if model_pth.exists() and model_pth.stat().st_size < 100 * 1024 * 1024:
    model_pth.unlink()
    print(f"Removed stale file ({model_pth.stat().st_size / 1024:.0f} KB), re-downloading...")

if not model_pth.exists():
    dl_ok = False
    model_dir.mkdir(parents=True, exist_ok=True)
    
    # Layer 1: Kaggle Dataset (GCP-local, no Xet, instant)
    if kaggle_path:
        src = Path(kaggle_path) / "mrf_0.1_shading_0.1_pca64_ep100_bth0.pth"
        if src.exists():
            shutil.copy(src, model_pth)
            print(f"Model weights: Kaggle Dataset ({model_pth.stat().st_size/(1024*1024):.0f} MB)")
            dl_ok = True
    
    # Layer 2: HF Hub via wget/requests (bypasses Xet Rust protocol)
    if not dl_ok:
        import torch as _torch
        model_url = ("https://huggingface.co/jacobthankgod4/smpl-model-garmentrec/"
                     "resolve/main/mrf_0.1_shading_0.1_pca64_ep100_bth0.pth?download=1")
        dl_ok = robust_download(model_url, model_pth, expected_mb=1100, desc="Model weights (~1.1GB)")
        # Post-download integrity check: try loading with torch
        if dl_ok:
            try:
                _chk = _torch.load(str(model_pth), map_location='cpu', weights_only=True)
                del _chk
                print(f"  Integrity: OK")
            except Exception as _e:
                print(f"  Integrity FAILED ({_e}), removing corrupt file...")
                model_pth.unlink(missing_ok=True)
                dl_ok = False

    # Layer 3: hf_hub_download (different CDN path, may bypass Xet 403)
    if not dl_ok:
        print("  Trying hf_hub_download (Layer 3)...")
        from huggingface_hub import hf_hub_download
        import subprocess as _sp, sys as _sys
        _env = {**{k: v for k, v in os.environ.items()}, 'HF_HUB_DISABLE_XET': '1'}
        try:
            _downloaded = hf_hub_download(
                repo_id='jacobthankgod4/smpl-model-garmentrec',
                filename='mrf_0.1_shading_0.1_pca64_ep100_bth0.pth',
                local_dir=str(model_dir),
                local_dir_use_symlinks=False,
            )
            if Path(_downloaded).exists() and Path(_downloaded).stat().st_size > 100 * 1024 * 1024:
                print(f"  hf_hub_download: OK ({Path(_downloaded).stat().st_size/(1024*1024):.0f} MB)")
                dl_ok = True
        except Exception as _e:
            print(f"  hf_hub_download failed: {_e}")
    
    if dl_ok:
        print(f"GarmentRec model: done ({model_pth.stat().st_size/(1024*1024):.0f} MB)")
    else:
        print("*** GARMENTREC MODEL WEIGHTS MISSING — reconstruction will fail ***")
else:
    size_mb = model_pth.stat().st_size / (1024*1024)
    print(f"GarmentRec model: cached ({size_mb:.0f} MB)")

# Final checks
tmps_target = GARMENT_REC_DIR / "data" / "tmps"
GARMENTS = ["T-shirt", "front_open_T-shirt", "Shirt", "front_open_Shirt", "Shorts", "Pants"]
if tmps_target.exists():
    all_ok = True
    for g in GARMENTS:
        obj = tmps_target / g / "garment_tmp.obj"
        ok = obj.exists() and obj.stat().st_size > 0
        if not ok:
            print(f"  *** {g}: MISSING garment_tmp.obj ***")
            all_ok = False
    if all_ok:
        print(f"tmps folder: OK ({len(list(tmps_target.iterdir()))} garment types, files verified)")
    else:
        print(f"\n*** tmps INCOMPLETE — see above. Re-run or check /tmp/garmentrec_assets_tmp/ ***")
else:
    print(f"\n*** tmps MISSING *** Expected at {tmps_target}")
    print("Check /tmp/garmentrec_assets_tmp/ for raw 7z contents.")

midpairs = GARMENT_REC_DIR / "data" / "midpairs.pkl"
dense_midpairs = GARMENT_REC_DIR / "data" / "dense_midpairs.pkl"
if midpairs.exists() and midpairs.stat().st_size > 100:
    print(f"midpairs.pkl: OK ({midpairs.stat().st_size / 1024:.1f} KB)")
else:
    print(f"*** midpairs.pkl MISSING at {midpairs} ***")
if dense_midpairs.exists():
    print(f"dense_midpairs.pkl: OK ({dense_midpairs.stat().st_size / 1024:.1f} KB)")
else:
    print(f"*** dense_midpairs.pkl MISSING at {dense_midpairs} ***")

# Fallback: download midpairs files from GitHub repo (they are git-tracked, not in 7z archive)
import urllib.request
for _pkl_name in ["midpairs.pkl", "dense_midpairs.pkl"]:
    _pkl_path = GARMENT_REC_DIR / "data" / _pkl_name
    if not _pkl_path.exists() or _pkl_path.stat().st_size < 100:
        _url = f"https://raw.githubusercontent.com/worryDes/GarmentRec/main/data/{_pkl_name}"
        print(f"Downloading {_pkl_name} from GitHub...")
        try:
            urllib.request.urlretrieve(_url, _pkl_path)
            print(f"  -> {_pkl_path} ({_pkl_path.stat().st_size / 1024:.0f} KB)")
        except Exception as _e:
            print(f"  *** FALLBACK FAILED for {_pkl_name}: {_e} ***")

# Check model weights
if not model_pth.exists() or model_pth.stat().st_size < 100 * 1024 * 1024:
    print(f"\n*** MODEL WEIGHTS {'MISSING' if not model_pth.exists() else 'TOO SMALL'} ***")
    print(f"Expected: {model_pth}")
else:
    print(f"Model weights: OK ({model_pth.stat().st_size / (1024*1024):.0f} MB)")

# SMPL model: multi-layer fallback
smpl_dir = GARMENT_REC_DIR / "smpl_pytorch" / "model"
smpl_dir.mkdir(parents=True, exist_ok=True)
smpl_path = smpl_dir / "neutral_smpl_with_cocoplus_reg.txt"
smpl_pkl = WD / "neutral_smpl_with_cocoplus_reg.pkl"
if not smpl_path.exists():
    smpl_ok = False
    # Layer 1: Kaggle Dataset
    if kaggle_path:
        src = Path(kaggle_path) / "neutral_smpl_with_cocoplus_reg.txt"
        if src.exists():
            shutil.copy(src, smpl_path)
            print(f"SMPL model: Kaggle Dataset")
            smpl_ok = True
    # Layer 2: HF Hub via wget/requests
    if not smpl_ok:
        smpl_url = ("https://huggingface.co/jacobthankgod4/smpl-model-garmentrec/"
                    "resolve/main/neutral_smpl_with_cocoplus_reg.txt")
        smpl_ok = robust_download(smpl_url, smpl_path, desc="SMPL model")
    if smpl_ok:
        print(f"SMPL model: OK ({smpl_path.stat().st_size/1024:.0f} KB)")
    else:
        print("*** SMPL MODEL MISSING ***")

if not smpl_path.exists() and smpl_pkl.exists():
    print("Converting SMPL .pkl to .txt (JSON)...")
    import pickle, json
    with open(smpl_pkl, 'rb') as f:
        pkl_data = pickle.load(f, encoding='latin1')
    # Convert chumpy/numpy/sparse → plain lists
    def to_serializable(v):
        if hasattr(v, 'r'):
            return v.r.tolist()
        if hasattr(v, 'toarray'):
            return v.toarray().tolist()
        if hasattr(v, 'tolist'):
            return v.tolist()
        if isinstance(v, (list, tuple)):
            return list(v)
        return v
    model_json = {k: to_serializable(v) for k, v in pkl_data.items()}
    with open(smpl_path, 'w') as f:
        json.dump(model_json, f)
    size_mb = smpl_path.stat().st_size / (1024*1024)
    print(f"SMPL model saved ({size_mb:.0f} MB)")

if not smpl_path.exists():
    print("\n*** SMPL MODEL NEEDED ***")
    print("Place neutral_smpl_with_cocoplus_reg.pkl or .txt at:")
    print(f"  {smpl_path}")
    print("Without this file, GarmentRec will fail at startup.\n")
else:
    size_mb = smpl_path.stat().st_size / (1024*1024)
    print(f"SMPL model ready ({size_mb:.0f} MB)")

# Patch deprecated numpy aliases in GarmentRec (NumPy 2.x removed np.float)
!sed -i 's/np\.float/float/g' {GARMENT_REC_DIR}/smpl_pytorch/SMPL.py 2>/dev/null || true
!sed -i 's/np\.float/float/g' {GARMENT_REC_DIR}/smpl_pytorch/util.py 2>/dev/null || true
# Patch SMPL J_regressor: transpose (24,6890) -> (6890,24) so matmul works
!sed -i "s/np_J_regressor = np.array(model\['J_regressor'\], dtype = np.float)/np_J_regressor = np.array(model['J_regressor'], dtype = np.float).T/" {GARMENT_REC_DIR}/smpl_pytorch/SMPL.py
print("Patched GarmentRec: np.float -> float (NumPy 2.x compat), J_regressor transposed")

# ---- pytorch3d shim: Phase 0-4 foundation (single-file backend) ----
import os as _os, sys as _sys, shutil as _shutil
_PT3D = _os.path.expanduser("~/pytorch3d")
print("[pytorch3d] Writing Phase 0-4 foundation to ~/pytorch3d/...")
if _os.path.exists(_PT3D):
    _shutil.rmtree(_PT3D)
for _sub in ["io", "structures", "renderer", "renderer/mesh", "ops", "loss", "common", "tools"]:
    _os.makedirs(f"{_PT3D}/{_sub}", exist_ok=True)

def _w(path, content):
    with open(path, "w") as f:
        f.write(content)

_SHIM_SRC = """
import os, torch, numpy as np
import torch.nn.functional as F
from types import SimpleNamespace
from typing import NamedTuple

__version__ = "0.0.0-shim"

class _C:
    def __getattr__(self, name):
        raise ImportError("pytorch3d._C not available")
C = _C()

def make_device(device):
    if isinstance(device, torch.device): return device
    if isinstance(device, str): return torch.device(device)
    return torch.device("cpu")

class TensorProperties:
    def __init__(self, dtype=torch.float32, device="cpu"):
        self.device = make_device(device); self.dtype = dtype
    def to(self, device):
        self.device = make_device(device)
        for k, v in self.__dict__.items():
            if isinstance(v, torch.Tensor): setattr(self, k, v.to(device))
        return self
    def clone(self):
        import copy; return copy.deepcopy(self)

def list_to_padded(x, pad_size=None, pad_value=0.0):
    if pad_size is None: pad_size = max(v.shape[0] for v in x)
    out = torch.full((len(x), pad_size, *x[0].shape[1:]), pad_value, dtype=x[0].dtype, device=x[0].device)
    for i, v in enumerate(x): out[i, :v.shape[0]] = v
    return out

def padded_to_list(x, num_items=None):
    if num_items is None: num_items = [x.shape[1]] * x.shape[0]
    return [x[i, :n] for i, n in enumerate(num_items)]

def list_to_packed(x): return torch.cat(x, dim=0)

def packed_to_list(x, num_items):
    out, offset = [], 0
    for n in num_items: out.append(x[offset:offset+n]); offset += n
    return out

def packed_to_padded(x, num_items, pad_size=None):
    return list_to_padded(packed_to_list(x, num_items), pad_size=pad_size)

def padded_to_packed(x, num_items=None):
    return list_to_packed(padded_to_list(x, num_items))

class Meshes:
    def __init__(self, verts=None, faces=None, textures=None, verts_normals=None):
        self._verts_list = list(verts) if isinstance(verts, (list, tuple)) else ([verts] if verts is not None else [])
        self._faces_list = list(faces) if isinstance(faces, (list, tuple)) else ([faces] if faces is not None else [])
        self._normals_list = list(verts_normals) if isinstance(verts_normals, (list, tuple)) else ([verts_normals] if verts_normals is not None else [])
        self._N = len(self._verts_list)
        self.textures = textures
        self._num_verts = [v.shape[0] for v in self._verts_list]
        self._num_faces = [f.shape[0] for f in self._faces_list]
        self.device = self._verts_list[0].device if self._verts_list else "cpu"
        self._clear()
    def _clear(self):
        for a in ["_vp","_fp","_ep","_vpi","_fpi","_vnp","_fnp","_fap","_vpad","_fpad","_vnpad"]: setattr(self, a, None)
    def __len__(self): return self._N
    def __getitem__(self, idx):
        if isinstance(idx, int):
            tex = self.textures[idx] if self.textures is not None and hasattr(self.textures, '__getitem__') else self.textures
            return Meshes(verts=[self._verts_list[idx].clone()], faces=[self._faces_list[idx].clone()], textures=tex)
        if isinstance(idx, slice):
            tex = self.textures[idx] if self.textures is not None and hasattr(self.textures, '__getitem__') else self.textures
            return Meshes(verts=[v.clone() for v in self._verts_list[idx]], faces=[f.clone() for f in self._faces_list[idx]], textures=tex)
        if isinstance(idx, torch.Tensor):
            idx = idx.tolist() if idx.dim() > 0 else [idx.item()]
            tex = self.textures[idx] if self.textures is not None and hasattr(self.textures, '__getitem__') else self.textures
            return Meshes(verts=[self._verts_list[i].clone() for i in idx], faces=[self._faces_list[i].clone() for i in idx], textures=tex)
        return self
    def split(self, split_sizes, dim=0):
        out, pos, tex = [], 0, self.textures
        for s in split_sizes:
            t = tex[pos:pos+s] if tex is not None and hasattr(tex, '__getitem__') else tex
            out.append(Meshes(verts=self._verts_list[pos:pos+s], faces=self._faces_list[pos:pos+s], textures=t))
            pos += s
        return out
    def _compute_packed(self):
        if self._vp is None and self._verts_list:
            self._vp = torch.cat(self._verts_list, dim=0)
            self._vpi = torch.cat([torch.full((n,), i, dtype=torch.int64) for i, n in enumerate(self._num_verts)])
        if self._fp is None and self._faces_list:
            off = 0; fl = []
            for i, f in enumerate(self._faces_list): fl.append(f + off); off += self._num_verts[i]
            self._fp = torch.cat(fl, dim=0)
            self._fpi = torch.cat([torch.full((n,), i, dtype=torch.int64) for i, n in enumerate(self._num_faces)])
    def verts_packed(self): self._compute_packed(); return self._vp
    def faces_packed(self): self._compute_packed(); return self._fp
    def _compute_padded(self):
        if self._vpad is None and self._verts_list:
            self._vpad = list_to_padded(self._verts_list)
        if self._fpad is None and self._faces_list:
            self._fpad = list_to_padded(self._faces_list, pad_value=-1)
    def verts_padded(self): self._compute_padded(); return self._vpad
    def faces_padded(self): self._compute_padded(); return self._fpad
    def verts_normals_padded(self):
        if self._vnpad is None:
            vn = self.verts_normals_packed()
            self._vnpad = packed_to_padded(vn, self._num_verts)
        return self._vnpad

    def verts_packed_to_mesh_idx(self): self._compute_packed(); return self._vpi
    def faces_packed_to_mesh_idx(self): self._compute_packed(); return self._fpi
    def num_verts_per_mesh(self): return self._num_verts
    def num_faces_per_mesh(self): return self._num_faces
    def edges_packed(self):
        if self._ep is None:
            f = self.faces_packed()
            if f is not None and f.numel() > 0:
                e = torch.cat([f[:,[0,1]], f[:,[1,2]], f[:,[2,0]]]).sort(dim=1)[0]
                self._ep = torch.unique(e, dim=0)
            else: self._ep = torch.zeros((0,2), dtype=torch.int64)
        return self._ep
    def verts_normals_packed(self):
        if self._vnp is None:
            if self._normals_list: self._vnp = torch.cat(self._normals_list, dim=0)
            else:
                f = self.faces_packed(); v = self.verts_packed()
                fn = torch.nn.functional.normalize(torch.cross(v[f[:,1]]-v[f[:,0]], v[f[:,2]]-v[f[:,0]]), dim=1)
                self._vnp = torch.zeros_like(v)
                self._vnp.index_add_(0, f[:,0], fn); self._vnp.index_add_(0, f[:,1], fn); self._vnp.index_add_(0, f[:,2], fn)
                self._vnp = torch.nn.functional.normalize(self._vnp, dim=1)
        return self._vnp
    def faces_normals_packed(self):
        if self._fnp is None:
            f = self.faces_packed(); v = self.verts_packed()
            self._fnp = torch.nn.functional.normalize(torch.cross(v[f[:,1]]-v[f[:,0]], v[f[:,2]]-v[f[:,0]]), dim=1)
        return self._fnp
    def to(self, device):
        self._verts_list = [v.to(device) for v in self._verts_list]
        self._faces_list = [f.to(device) for f in self._faces_list]
        self.device = device; self._clear(); return self
    def cuda(self): return self.to("cuda")
    def cpu(self): return self.to("cpu")
    def clone(self):
        tex = self.textures.clone() if self.textures is not None and hasattr(self.textures, 'clone') else self.textures
        nl = [n.clone() for n in self._normals_list] if self._normals_list else None
        return Meshes(verts=[v.clone() for v in self._verts_list], faces=[f.clone() for f in self._faces_list], textures=tex, verts_normals=nl)
    def detach(self):
        return Meshes(verts=[v.detach() for v in self._verts_list], faces=[f.detach() for f in self._faces_list])
    def offset_verts(self, off):
        idx = 0
        nl = []
        for i in range(self._N):
            n = self._num_verts[i]; nl.append(self._verts_list[i] + off[idx:idx+n]); idx += n
        return Meshes(verts=nl, faces=[f.clone() for f in self._faces_list], textures=self.textures, verts_normals=[v.clone() for v in self._normals_list] if self._normals_list else None)
    def offset_verts_(self, off):
        idx = 0
        for i in range(self._N):
            n = self._num_verts[i]; self._verts_list[i] = self._verts_list[i] + off[idx:idx+n]; idx += n
        self._clear(); return self
    def update_padded(self, new_vp):
        for i in range(self._N): self._verts_list[i] = new_vp[i, :self._num_verts[i]]
        self._clear(); return self
    def sample_textures(self, fragments):
        if self.textures is not None and hasattr(self.textures, "sample_textures"):
            return self.textures.sample_textures(fragments)
        if fragments is not None and fragments.pix_to_face is not None:
            N, H, W, K = fragments.pix_to_face.shape
            return torch.zeros((N, H, W, K, 3))
        return torch.zeros((self._N, 512, 512, 3))

class Pointclouds:
    def __init__(self, points=None, **kwargs):
        self.points_list = [points] if isinstance(points, torch.Tensor) else (list(points) if points is not None else [])
    def __len__(self): return len(self.points_list)

class ObjLoadResult(NamedTuple):
    verts: torch.Tensor; faces: torch.Tensor; aux: SimpleNamespace

class FacesData:
    __slots__ = ('verts_idx', 'textures_idx', 'normals_idx')
    def __init__(self, verts_idx, textures_idx=None, normals_idx=None):
        self.verts_idx = verts_idx
        self.textures_idx = textures_idx if textures_idx is not None else verts_idx
        self.normals_idx = normals_idx if normals_idx is not None else verts_idx

def load_obj(path, load_textures=True, **kwargs):
    path = str(path); import trimesh; mesh = trimesh.load(path, force="mesh")
    verts = torch.tensor(mesh.vertices, dtype=torch.float32)
    faces = torch.tensor(mesh.faces, dtype=torch.int64)
    vu, fu, ti, mc = None, None, None, None
    if hasattr(mesh.visual, "uv") and mesh.visual.uv is not None:
        vu = torch.tensor(mesh.visual.uv, dtype=torch.float32)
        fu = torch.tensor(mesh.faces, dtype=torch.int64)
        if hasattr(mesh.visual, "material") and mesh.visual.material:
            mat = mesh.visual.material
            if hasattr(mat, "image") and mat.image is not None:
                import PIL.Image as PILI; ti = [PILI.fromarray(np.array(mat.image)).convert("RGB")]
            if hasattr(mat, "main_color") and mat.main_color is not None:
                c = mat.main_color[:3] if len(mat.main_color) >= 3 else [1,1,1]
                mc = SimpleNamespace(ambient_color=torch.tensor(c), diffuse_color=torch.tensor(c), specular_color=torch.tensor([0.,0.,0.]), shininess=0.)
    faces_d = faces if isinstance(faces, torch.Tensor) else torch.tensor(faces) if faces is not None else torch.zeros((0,3), dtype=torch.int64)
    fu_d = fu if fu is not None else faces_d
    return ObjLoadResult(verts=verts, faces=FacesData(verts_idx=faces_d, textures_idx=fu_d), aux=SimpleNamespace(verts_uvs=vu, faces_uvs=fu, texture_images=ti, material_colors=mc))

def load_objs_as_meshes(obj_files, load_textures=True, **kwargs):
    vl, fl = [], []
    for f in obj_files:
        r = load_obj(f, load_textures=load_textures, **kwargs); vl.append(r.verts); fl.append(r.faces)
    return Meshes(verts=vl, faces=fl)

def save_obj(f, verts, faces, verts_uvs=None, faces_uvs=None, texture_map=None, **kwargs):
    f = str(f)
    if isinstance(verts, torch.Tensor): verts = verts.detach().cpu().numpy()
    elif not isinstance(verts, np.ndarray): verts = np.array(verts)
    if isinstance(faces, torch.Tensor): faces = faces.detach().cpu().numpy()
    elif not isinstance(faces, np.ndarray): faces = np.array(faces)
    import trimesh; mesh = trimesh.Trimesh(vertices=verts, faces=faces)
    if verts_uvs is not None:
        if isinstance(verts_uvs, torch.Tensor): verts_uvs = verts_uvs.detach().cpu().numpy()
        mesh.visual = trimesh.visual.TextureVisuals(uv=verts_uvs)
    mesh.export(f, file_type="obj")
    if texture_map is not None:
        mtl_path = f.replace(".obj", ".mtl"); png_path = f.replace(".obj", ".png")
        if isinstance(texture_map, torch.Tensor):
            texture_map = texture_map.detach().cpu()
            import PIL.Image as PILI
            arr = (texture_map.numpy() * 255).astype(np.uint8)
            if arr.ndim == 3:
                if arr.shape[-1] in (3, 4): PILI.fromarray(arr).save(png_path)
                elif arr.shape[0] in (3, 4): PILI.fromarray(arr.transpose(1,2,0)).save(png_path)
        with open(mtl_path, "w") as mf:
            mf.write("newmtl material0\\n")
            mf.write(f"map_Kd {os.path.basename(png_path)}\\n")

class RasterizationSettings:
    def __init__(self, image_size=512, blur_radius=0.0, faces_per_pixel=1, bin_size=None,
                 max_faces_per_bin=None, perspective_correct=False, clip_barycentric_coords=False,
                 cull_backfaces=False, z_clip_value=None, cull_to_frustum=False):
        self.image_size = image_size; self.blur_radius = blur_radius; self.faces_per_pixel = faces_per_pixel
        self.bin_size = bin_size; self.max_faces_per_bin = max_faces_per_bin
        self.perspective_correct = perspective_correct; self.clip_barycentric_coords = clip_barycentric_coords
        self.cull_backfaces = cull_backfaces; self.z_clip_value = z_clip_value; self.cull_to_frustum = cull_to_frustum

class Fragments:
    def __init__(self, pix_to_face=None, zbuf=None, bary_coords=None, dists=None):
        self.pix_to_face = pix_to_face; self.zbuf = zbuf; self.bary_coords = bary_coords; self.dists = dists
    def detach(self):
        return Fragments(*(x.detach() if x is not None else None for x in [self.pix_to_face, self.zbuf, self.bary_coords, self.dists]))

def rasterize_meshes(meshes, raster_settings=None, image_size=None, blur_radius=0.0, faces_per_pixel=1, **kwargs):
    s = image_size or (getattr(raster_settings, "image_size", 512) if raster_settings else 512)
    if isinstance(s, (list, tuple)): H, W = s
    else: H = W = s
    fp = faces_per_pixel if faces_per_pixel is not None else getattr(raster_settings, "faces_per_pixel", 1) if raster_settings else 1
    br = blur_radius if blur_radius is not None else getattr(raster_settings, "blur_radius", 0.0) if raster_settings else 0.0
    cull = getattr(raster_settings, "cull_backfaces", False) if raster_settings else False
    persp_correct = getattr(raster_settings, "perspective_correct", False) if raster_settings else False
    clip_bary = getattr(raster_settings, "clip_barycentric_coords", False) if raster_settings else False
    n = len(meshes) if hasattr(meshes, "__len__") else 1
    d = meshes.device if hasattr(meshes, "device") else "cpu"
    pix_to_face = torch.full((n, H, W, fp), -1, dtype=torch.int64, device=d)
    zbuf = torch.full((n, H, W, fp), 1e10, dtype=torch.float32, device=d)
    bary = torch.zeros((n, H, W, fp, 3), dtype=torch.float32, device=d)
    dists = torch.full((n, H, W, fp), 1e10, dtype=torch.float32, device=d)
    for mi in range(n):
        if not hasattr(meshes, "_verts_list") or mi >= len(meshes._verts_list): continue
        verts = meshes._verts_list[mi]; faces = meshes._faces_list[mi]
        if verts is None or faces is None or verts.numel() == 0 or faces.numel() == 0: continue
        v0 = verts[faces[:, 0]]; v1 = verts[faces[:, 1]]; v2 = verts[faces[:, 2]]
        if cull:
            normal = torch.cross(v1 - v0, v2 - v0, dim=1)
            keep = normal[:, 2] <= 0
            v0, v1, v2, fi_all = v0[keep], v1[keep], v2[keep], torch.where(keep)[0]
        else:
            fi_all = torch.arange(faces.shape[0], device=d)
        # Sort faces far-to-near (Painter algorithm)
        fz = -(v0[:, 2] + v1[:, 2] + v2[:, 2]) / 3.0  # positive = closer
        order = torch.argsort(fz, descending=False)  # far first
        for idx in order:
            fi = fi_all[idx].item() if fi_all.dim() > 0 else idx
            fv0, fv1, fv2 = v0[idx], v1[idx], v2[idx]
            xs = torch.stack([fv0[0], fv1[0], fv2[0]])
            ys = torch.stack([fv0[1], fv1[1], fv2[1]])
            xmin = max(0, int(torch.floor(xs.min()).item()))
            xmax = min(W-1, int(torch.ceil(xs.max()).item()))
            ymin = max(0, int(torch.floor(ys.min()).item()))
            ymax = min(H-1, int(torch.ceil(ys.max()).item()))
            if xmin >= xmax or ymin >= ymax: continue
            gy, gx = torch.meshgrid(torch.arange(ymin, ymax+1, device=d), torch.arange(xmin, xmax+1, device=d), indexing="ij")
            px = gx.float() + 0.5; py = gy.float() + 0.5
            e0x = fv1[0]-fv0[0]; e0y = fv1[1]-fv0[1]
            e1x = fv2[0]-fv1[0]; e1y = fv2[1]-fv1[1]
            e2x = fv0[0]-fv2[0]; e2y = fv0[1]-fv2[1]
            area2 = torch.abs(e0x*(fv2[1]-fv0[1]) - e0y*(fv2[0]-fv0[0])) + 1e-10
            c0 = e0x*(py-fv0[1]) - e0y*(px-fv0[0])
            c1 = e1x*(py-fv1[1]) - e1y*(px-fv1[0])
            c2 = e2x*(py-fv2[1]) - e2y*(px-fv2[0])
            inside = (c0 >= 0) & (c1 >= 0) & (c2 >= 0)
            if not inside.any(): continue
            b0 = c0 / area2; b1 = c1 / area2; b2 = 1.0 - b0 - b1
            zvals = -(b0*fv0[2] + b1*fv1[2] + b2*fv2[2])
            iy, ix = gy[inside], gx[inside]
            iz, ib0, ib1, ib2 = zvals[inside], b0[inside], b1[inside], b2[inside]
            # Perspective correction: multiply bary by 1/z and renormalize
            b0_corr = b0.clone(); b1_corr = b1.clone(); b2_corr = b2.clone()
            # We apply perspective_correct per-pixel after bary computation
            # Store raw bary coords for now, correct before z-buffer
            rcp_z0 = 1.0 / (-fv0[2].clamp(min=1e-6))
            rcp_z1 = 1.0 / (-fv1[2].clamp(min=1e-6))
            rcp_z2 = 1.0 / (-fv2[2].clamp(min=1e-6))
            bary_scale = (b0 * rcp_z0 + b1 * rcp_z1 + b2 * rcp_z2).clamp(min=1e-10)
            if fp == 1:
                # Fast vectorized path (no Python .item() calls)
                # Apply perspective correction to barycentric coords
                # perspective_correct=True: multiply bary by 1/z, renormalize
                b0_pc = ib0 * rcp_z0 / ((ib0 * rcp_z0 + ib1 * rcp_z1 + ib2 * rcp_z2).clamp(min=1e-10))
                b1_pc = ib1 * rcp_z1 / ((ib0 * rcp_z0 + ib1 * rcp_z1 + ib2 * rcp_z2).clamp(min=1e-10))
                b2_pc = ib2 * rcp_z2 / ((ib0 * rcp_z0 + ib1 * rcp_z1 + ib2 * rcp_z2).clamp(min=1e-10))
                # Clip barycentric coords if requested
                cb0 = b0_pc.clamp(min=0); cb1 = b1_pc.clamp(min=0); cb2 = b2_pc.clamp(min=0)
                cb_sum = (cb0 + cb1 + cb2).clamp(min=1e-10)
                cb0 = cb0 / cb_sum; cb1 = cb1 / cb_sum; cb2 = cb2 / cb_sum
                curr_z = zbuf[mi, iy, ix, 0]
                closer = iz < curr_z
                if not closer.any(): continue
                cy, cx = iy[closer], ix[closer]
                cz, cb0, cb1, cb2 = iz[closer], cb0[closer], cb1[closer], cb2[closer]
                zbuf[mi, cy, cx, 0] = cz
                pix_to_face[mi, cy, cx, 0] = fi
                bary[mi, cy, cx, 0, 0] = cb0
                bary[mi, cy, cx, 0, 1] = cb1
                bary[mi, cy, cx, 0, 2] = cb2
                dists[mi, cy, cx, 0] = 0.0
            else:
                # Fallback: insertion sort (fp>1 rare)
                for pi in range(iy.shape[0]):
                    pyi, pxi, pz = iy[pi].item(), ix[pi].item(), iz[pi].item()
                    b0_i = ib0[pi].item(); b1_i = ib1[pi].item(); b2_i = ib2[pi].item()
                    # Perspective correct
                    pz0 = -fv0[2].item(); pz1 = -fv1[2].item(); pz2 = -fv2[2].item()
                    r0 = 1.0 / max(pz0, 1e-6); r1 = 1.0 / max(pz1, 1e-6); r2 = 1.0 / max(pz2, 1e-6)
                    denom = b0_i * r0 + b1_i * r1 + b2_i * r2
                    if denom > 1e-10:
                        cb0_i = b0_i * r0 / denom
                        cb1_i = b1_i * r1 / denom
                        cb2_i = b2_i * r2 / denom
                    else:
                        cb0_i, cb1_i, cb2_i = b0_i, b1_i, b2_i
                    # Clip negative bary
                    if cb0_i < 0: cb0_i = 0.0
                    if cb1_i < 0: cb1_i = 0.0
                    if cb2_i < 0: cb2_i = 0.0
                    csum = cb0_i + cb1_i + cb2_i
                    if csum > 1e-10:
                        cb0_i /= csum; cb1_i /= csum; cb2_i /= csum
                    slot = 0
                    while slot < fp and zbuf[mi, pyi, pxi, slot] <= pz: slot += 1
                    if slot >= fp: continue
                    if slot < fp - 1:
                        zbuf[mi, pyi, pxi, slot+1:] = zbuf[mi, pyi, pxi, slot:-1].clone()
                        pix_to_face[mi, pyi, pxi, slot+1:] = pix_to_face[mi, pyi, pxi, slot:-1].clone()
                        bary[mi, pyi, pxi, slot+1:] = bary[mi, pyi, pxi, slot:-1].clone()
                    zbuf[mi, pyi, pxi, slot] = pz
                    pix_to_face[mi, pyi, pxi, slot] = int(fi)
                    bary[mi, pyi, pxi, slot, 0] = cb0_i
                    bary[mi, pyi, pxi, slot, 1] = cb1_i
                    bary[mi, pyi, pxi, slot, 2] = cb2_i
                    dists[mi, pyi, pxi, slot] = 0.0
    zbuf[zbuf == 1e10] = -1.0
    dists[dists == 1e10] = -1.0
    return pix_to_face, zbuf, bary, dists
class MeshRasterizer:
    def __init__(self, cameras=None, raster_settings=None, **kwargs):
        self.cameras = cameras; self.raster_settings = raster_settings or RasterizationSettings()
    def forward(self, meshes, **kwargs):
        if meshes is None or len(meshes) == 0:
            return Fragments()
        # Transform vertices to screen space
        if self.cameras is not None:
            rs = self.raster_settings
            im_size = getattr(rs, "image_size", 512) if rs else 512
            # Transform each mesh's vertices to screen space
            new_verts = []
            for v in meshes._verts_list:
                sv = self.cameras.transform_points_screen(v, image_size=im_size)
                new_verts.append(sv)
            screen_meshes = Meshes(verts=new_verts, faces=meshes._faces_list)
        else:
            screen_meshes = meshes
        pf, zb, bc, ds = rasterize_meshes(screen_meshes, raster_settings=self.raster_settings)
        return Fragments(pix_to_face=pf, zbuf=zb, bary_coords=bc, dists=ds)
    def __call__(self, meshes, **kwargs): return self.forward(meshes, **kwargs)
class PointLights(TensorProperties):
    def __init__(self, device="cpu", ambient_color=((0.5,),), diffuse_color=((0.3,),), specular_color=((0.2,),), location=((0.,0.,1.),), **kwargs):
        super().__init__(device=device)
        self.ambient_color = torch.tensor(ambient_color, device=device)
        self.diffuse_color = torch.tensor(diffuse_color, device=device)
        self.specular_color = torch.tensor(specular_color, device=device)
        self.location = torch.tensor(location, device=device)

class AmbientLights(TensorProperties):
    def __init__(self, device="cpu", ambient_color=((0.5,),), **kwargs):
        super().__init__(device=device); self.ambient_color = torch.tensor(ambient_color, device=device)

class DirectionalLights(TensorProperties):
    def __init__(self, device="cpu", **kwargs):
        super().__init__(device=device)
        for k, v in kwargs.items():
            if isinstance(v, (tuple, list)): setattr(self, k, torch.tensor(v, device=device))
            else: setattr(self, k, v)

class PerspectiveCameras(TensorProperties):
    def __init__(self, device="cpu", R=None, T=None, focal_length=None, principal_point=None, in_ndc=True, image_size=None, **kwargs):
        super().__init__(device=device)
        self._R = torch.tensor(R, device=device) if R is not None else torch.eye(3, device=device).unsqueeze(0)
        self._T = torch.tensor(T, device=device) if T is not None else torch.zeros(1, 3, device=device)
        self.focal_length = torch.tensor(focal_length, device=device) if focal_length is not None else torch.tensor([[270.,270.]], device=device)
        self.principal_point = torch.tensor(principal_point, device=device) if principal_point is not None else torch.tensor([[270.,270.]], device=device)
        self.image_size = torch.tensor(image_size, device=device) if image_size is not None else torch.tensor([[540,540]], device=device)
        self.in_ndc = in_ndc
    def get_world_to_view_transform(self):
        t = SimpleNamespace(); R, T = self._R, self._T
        t.transform_points = lambda pts: (pts @ R.squeeze(0).T + T.squeeze(0)) if R is not None and T is not None else (pts if pts.dim() == 2 else pts.squeeze(0))
        t.get_matrix = lambda: torch.cat([R, T.unsqueeze(-1)], dim=-1) if R is not None else torch.eye(4).unsqueeze(0); return t
    def get_ndc_camera_transform(self):
        t = SimpleNamespace(); t.transform_points = lambda pts: pts; return t
    def transform_points(self, points, **kwargs): return points @ self._R.transpose(-2,-1) + self._T
    def transform_points_screen(self, points, image_size=None, **kwargs):
        # Project 3D points to screen space with perspective.
        if points.dim() == 2: pts_3d = points.unsqueeze(0)
        else: pts_3d = points
        pts_cam = pts_3d @ self._R.transpose(-2, -1) + self._T.unsqueeze(1)
        fx = self.focal_length[..., 0:1].unsqueeze(-1)
        fy = self.focal_length[..., 1:2].unsqueeze(-1)
        px = self.principal_point[..., 0:1].unsqueeze(-1)
        py = self.principal_point[..., 1:2].unsqueeze(-1)
        z = pts_cam[..., 2:3].clamp(min=1e-6)
        x_proj = pts_cam[..., 0:1] * fx / z + px
        y_proj = pts_cam[..., 1:2] * fy / z + py
        if image_size is not None:
            if isinstance(image_size, (list, tuple)): h, w = image_size
            else: h = w = image_size
            h = torch.tensor(h, dtype=torch.float32, device=self.device)
            w = torch.tensor(w, dtype=torch.float32, device=self.device)
            x_screen = w * (x_proj + 1.0) / 2.0
            y_screen = h * (1.0 - (y_proj + 1.0) / 2.0)
        else:
            x_screen = x_proj
            y_screen = y_proj
        out = torch.cat([x_screen, y_screen, -z], dim=-1)
        if points.dim() == 2: return out[0]
        return out
    def get_camera_center(self): return -self._R.transpose(-2,-1) @ self._T.unsqueeze(-1)
    def is_perspective(self): return True
    def get_znear(self): return torch.tensor(0.1, device=self.device)
    def get_zfar(self): return torch.tensor(100.0, device=self.device)

class FoVPerspectiveCameras(PerspectiveCameras): pass

class BlendParams:
    def __init__(self, sigma=1e-4, gamma=1e-4, background_color=(0.,0.,0.)):
        self.sigma = sigma; self.gamma = gamma; self.background_color = background_color

class TexturesBase:
    def isempty(self): return False
    def to(self, device): return self
    def clone(self): return self
    def detach(self): return self

class TexturesVertex(TexturesBase):
    def __init__(self, verts_features=None, **kwargs):
        self._list = list(verts_features) if isinstance(verts_features, (list, tuple)) else ([verts_features] if verts_features is not None else [])
    def verts_features_padded(self):
        if not self._list: return torch.zeros((1,0,3))
        mv = max(v.shape[0] for v in self._list); out = torch.zeros((len(self._list), mv, 3))
        for i,v in enumerate(self._list): out[i,:v.shape[0]] = v; return out
    def sample_textures(self, fragments): return torch.zeros((len(self._list) if self._list else 1, 512, 512, 3))

class TexturesUV(TexturesBase):
    def __init__(self, maps=None, faces_uvs=None, verts_uvs=None, **kwargs):
        if isinstance(maps, torch.Tensor) and maps.dim() == 3: self._maps_padded = maps.unsqueeze(0)
        else: self._maps_padded = torch.stack(list(maps)) if isinstance(maps, (list,tuple)) and maps else (maps if maps is not None else torch.zeros(1,1,1,3))
        self.fu_list = list(faces_uvs) if isinstance(faces_uvs, (list,tuple)) else ([faces_uvs] if faces_uvs is not None else [])
        self.vu_list = list(verts_uvs) if isinstance(verts_uvs, (list,tuple)) else ([verts_uvs] if verts_uvs is not None else [])
    def __getitem__(self, idx):
        if isinstance(idx, int):
            m = self._maps_padded[idx:idx+1] if self._maps_padded is not None and self._maps_padded.dim() == 4 else self._maps_padded
            vu = [self.vu_list[idx].clone()] if idx < len(self.vu_list) else []
            fu = [self.fu_list[idx].clone()] if idx < len(self.fu_list) else []
            return TexturesUV(maps=m, faces_uvs=fu, verts_uvs=vu)
        if isinstance(idx, slice):
            m = self._maps_padded[idx] if self._maps_padded is not None and self._maps_padded.dim() == 4 else self._maps_padded
            return TexturesUV(maps=m, faces_uvs=self.fu_list[idx], verts_uvs=self.vu_list[idx])
        if isinstance(idx, (list, tuple)):
            m = self._maps_padded[idx] if self._maps_padded is not None and self._maps_padded.dim() == 4 and len(idx) <= self._maps_padded.shape[0] else self._maps_padded
            return TexturesUV(maps=m, faces_uvs=[self.fu_list[i].clone() for i in idx], verts_uvs=[self.vu_list[i].clone() for i in idx])
        return self
    def verts_uvs_padded(self):
        if not self.vu_list: return torch.zeros((1,0,2))
        mv = max(v.shape[0] for v in self.vu_list); out = torch.zeros((len(self.vu_list), mv, 2))
        for i,v in enumerate(self.vu_list): out[i,:v.shape[0]] = v[...,:2]; return out
    def faces_uvs_padded(self):
        if not self.fu_list: return torch.zeros((1,0,3), dtype=torch.int64)
        mf = max(f.shape[0] for f in self.fu_list); out = torch.zeros((len(self.fu_list), mf, 3), dtype=torch.int64)
        for i,f in enumerate(self.fu_list): out[i,:f.shape[0]] = f; return out
    def maps_padded(self): return self._maps_padded if hasattr(self, '_maps_padded') else self.maps_padded
    def sample_textures(self, fragments):
        if fragments is None or fragments.pix_to_face is None:
            maps = self.maps_padded
            return torch.zeros((maps.shape[0] if maps.dim() == 4 else 1, 1, 1, 3), device=maps.device)
        pix_to_face = fragments.pix_to_face; bary_coords = fragments.bary_coords
        N, H, W, K = pix_to_face.shape; device = pix_to_face.device
        maps = self._maps_padded.to(device)
        C = maps.shape[-1] if maps.dim() == 4 else (maps.shape[0] if maps.dim() == 3 else 3)
        out = torch.zeros(N, H, W, K, C, device=device)
        for i in range(N):
            if i >= len(self.vu_list) or i >= len(self.fu_list):
                continue
            vui = self.vu_list[i].to(device)
            fui = self.fu_list[i].to(device)
            fi = pix_to_face[i].clamp(min=0).long()
            uv_idx = fui[fi]
            uv_coords = vui[uv_idx.long()]
            pixel_uv = (uv_coords * bary_coords[i].unsqueeze(-1)).sum(dim=-2)
            grid = pixel_uv * 2 - 1
            map_i = maps[i] if maps.dim() == 4 else maps
            if map_i.dim() == 3:
                if map_i.shape[0] <= 4 and map_i.shape[-1] > 4:
                    map_i = map_i.permute(2, 0, 1)
            if K == 1:
                sampled = F.grid_sample(
                    map_i.unsqueeze(0), grid.unsqueeze(0),
                    mode='bilinear', padding_mode='border', align_corners=False
                )
                out[i, :, :, 0, :C] = sampled.squeeze(0).permute(1, 2, 0)
            else:
                for k in range(K):
                    sampled = F.grid_sample(
                        map_i.unsqueeze(0), grid[:, :, k:k+1, :],
                        mode='bilinear', padding_mode='border', align_corners=False
                    )
                    out[i, :, :, k, :C] = sampled.squeeze(0).permute(1, 2, 0)
        return out
    def clone(self):
        return TexturesUV(maps=self._maps_padded.clone(), faces_uvs=[f.clone() for f in self.fu_list], verts_uvs=[v.clone() for v in self.vu_list])
    def detach(self):
        return TexturesUV(maps=self._maps_padded.detach(), faces_uvs=[f.detach() for f in self.fu_list], verts_uvs=[v.detach() for v in self.vu_list])
    def to(self, device):
        self.maps_padded = self._maps_padded.to(device)
        self.fu_list = [f.to(device) for f in self.fu_list]; self.vu_list = [v.to(device) for v in self.vu_list]; return self

class TexturesAtlas(TexturesBase):
    def __init__(self, atlas=None, **kwargs): self.atlas = atlas
    def sample_textures(self, fragments): return torch.zeros((self.atlas.shape[0] if self.atlas is not None else 1, 512, 512, 3))

def softmax_rgb_blend(colors, fragments, blend_params, znear=-1.0, zfar=1.0): return colors[..., :3]
def sigmoid_alpha_blend(colors, fragments, blend_params):
    alpha = (fragments.pix_to_face >= 0).float() if fragments.pix_to_face is not None else torch.zeros(1)
    return torch.cat([colors[..., :3], alpha], dim=-1)
def hard_rgb_blend(colors, fragments, blend_params): return colors[..., :3]

class ShaderBase:
    def __init__(self, device="cpu", cameras=None, lights=None, materials=None, blend_params=None):
        self.device = make_device(device); self.cameras = cameras; self.lights = lights
        self.materials = materials; self.blend_params = blend_params or BlendParams()
    def to(self, device): self.device = make_device(device); return self

class SoftPhongShader(ShaderBase):
    def __init__(self, device="cpu", cameras=None, lights=None, materials=None, blend_params=None, **kwargs):
        super().__init__(device, cameras, lights, materials, blend_params)
    def forward(self, fragments, meshes, **kwargs):
        if fragments.pix_to_face is None: return torch.zeros((1,512,512,4), device=self.device)
        h,w = fragments.pix_to_face.shape[1], fragments.pix_to_face.shape[2]
        texels = meshes.sample_textures(fragments)
        colors = phong_shading(meshes, fragments, texels, self.lights, self.cameras, self.materials)
        return softmax_rgb_blend(colors, fragments, self.blend_params)
    def __call__(self, fragments, meshes, **kwargs): return self.forward(fragments, meshes, **kwargs)

class SoftSilhouetteShader(ShaderBase):
    def __init__(self, device="cpu", blend_params=None, **kwargs): super().__init__(device=device, blend_params=blend_params)
    def forward(self, fragments, meshes, **kwargs):
        if fragments.pix_to_face is None: return torch.zeros((1,512,512,4), device=self.device)
        h,w = fragments.pix_to_face.shape[1], fragments.pix_to_face.shape[2]
        b = fragments.pix_to_face.shape[0]; alpha = (fragments.pix_to_face >= 0).float().max(dim=-1, keepdim=True)[0]
        return torch.cat([torch.ones((b,h,w,3), device=self.device), alpha.to(self.device)], dim=-1)
    def __call__(self, fragments, meshes, **kwargs): return self.forward(fragments, meshes, **kwargs)

class HardPhongShader(ShaderBase):
    def __init__(self, device="cpu", cameras=None, lights=None, materials=None, blend_params=None, **kwargs):
        super().__init__(device, cameras, lights, materials, blend_params)

class MeshRenderer:
    def __init__(self, rasterizer=None, shader=None, **kwargs):
        self.rasterizer = rasterizer; self.shader = shader
    def forward(self, meshes, **kwargs):
        fragments = self.rasterizer(meshes) if self.rasterizer else Fragments()
        return self.shader(fragments, meshes) if self.shader else torch.zeros((1,512,512,4))
    def __call__(self, meshes, **kwargs): return self.forward(meshes, **kwargs)
    def to(self, device):
        if self.rasterizer and hasattr(self.rasterizer.cameras, "to"): self.rasterizer.cameras.to(device)
        return self

class MeshRendererWithFragments(MeshRenderer):
    def forward(self, meshes, **kwargs):
        fragments = self.rasterizer(meshes) if self.rasterizer else Fragments()
        images = self.shader(fragments, meshes) if self.shader else torch.zeros((1,512,512,4))
        return images, fragments

def phong_shading(meshes, fragments, texels, lights, cameras, materials, **kwargs):
    if lights is None or cameras is None:
        return texels
    pix_to_face = fragments.pix_to_face
    bary_coords = fragments.bary_coords
    verts_normals = meshes.verts_normals_packed()
    faces = meshes.faces_packed()
    face_normals = verts_normals[faces]
    pixel_normals = interpolate_face_attributes(pix_to_face, bary_coords, face_normals)
    pixel_normals = F.normalize(pixel_normals, dim=-1)
    # Camera center for specular
    cam_center = cameras.get_camera_center() if hasattr(cameras, 'get_camera_center') else None
    # Ambient
    ambient = lights.ambient_color.to(pixel_normals.device) if hasattr(lights, 'ambient_color') else 0.0
    while ambient.dim() < 4: ambient = ambient.unsqueeze(0)
    # Diffuse
    diffuse = lights.diffuse_color.to(pixel_normals.device) if hasattr(lights, 'diffuse_color') else 0.0
    while diffuse.dim() < 4: diffuse = diffuse.unsqueeze(0)
    # Specular
    specular = lights.specular_color.to(pixel_normals.device) if hasattr(lights, 'specular_color') else 0.0
    while specular.dim() < 4: specular = specular.unsqueeze(0)
    light_loc = lights.location.to(pixel_normals.device) if hasattr(lights, 'location') else None
    # Interpolate camera-space positions
    if cam_center is not None and light_loc is not None:
        verts = meshes.verts_packed()
        face_pos = verts[faces]
        pixel_pos = interpolate_face_attributes(pix_to_face, bary_coords, face_pos)
        to_light = F.normalize(light_loc - pixel_pos, dim=-1)
        diff = diffuse * torch.clamp((to_light * pixel_normals).sum(dim=-1, keepdim=True), 0.0, 1.0)
        if cam_center is not None and hasattr(materials, 'shininess'):
            to_view = F.normalize(cam_center - pixel_pos, dim=-1)
            half = F.normalize(to_light + to_view, dim=-1)
            spec = specular * torch.pow(
                torch.clamp((half * pixel_normals).sum(dim=-1, keepdim=True), 0.0, 1.0),
                getattr(materials, 'shininess', 0.0)
            )
        else:
            spec = torch.zeros_like(texels)
    else:
        diff = torch.zeros_like(texels)
        spec = torch.zeros_like(texels)
    lighting = ambient + diff + spec
    # Match texels dims (texels may be 5D: N,H,W,K,C, lighting may be 4D: N,H,W,C)
    while lighting.dim() < texels.dim():
        lighting = lighting.unsqueeze(-2)
    return texels * lighting[..., :texels.shape[-1]]

def subdivide_meshes(meshes, **kwargs): return meshes

def knn_points(query, cloud, K=1, **kwargs):
    d = torch.zeros((query.shape[0], query.shape[1], K), device=query.device)
    i = torch.zeros((query.shape[0], query.shape[1], K), dtype=torch.int64, device=query.device)
    r = SimpleNamespace(); r.dists = d; r.idx = i; return r

def interpolate_face_attributes(pix_to_face, bary_coords, face_attributes):
    if pix_to_face is None or pix_to_face.numel() == 0: return torch.zeros((1,512,512,3))
    fi = pix_to_face.clamp(min=0); g = face_attributes[fi]
    return (g * bary_coords.unsqueeze(-1)).sum(dim=-2)

def mesh_face_areas_normals(verts_packed, faces_packed):
    v0 = verts_packed[faces_packed[:,0]]; v1 = verts_packed[faces_packed[:,1]]; v2 = verts_packed[faces_packed[:,2]]
    n = torch.nn.functional.normalize(torch.cross(v1-v0, v2-v0), dim=1)
    a = torch.norm(torch.cross(v1-v0, v2-v0), dim=1) * 0.5; return a, n

class SubdivideMeshes:
    def __init__(self, **kwargs): pass
    @staticmethod
    def _subdivide_mesh(verts, faces):
        v = verts.numpy() if isinstance(verts, torch.Tensor) else verts
        f = faces.numpy() if isinstance(faces, torch.Tensor) else faces
        import trimesh; tm = trimesh.Trimesh(vertices=v, faces=f)
        tm = tm.subdivide()
        return torch.tensor(tm.vertices, dtype=torch.float32), torch.tensor(tm.faces, dtype=torch.int64)
    def forward(self, meshes, **kwargs):
        if meshes is None or len(meshes) == 0: return meshes
        if not hasattr(meshes, "_verts_list") or not meshes._verts_list: return meshes
        nl, fl = [], []
        for v, f in zip(meshes._verts_list, meshes._faces_list):
            nv, nf = self._subdivide_mesh(v, f); nl.append(nv); fl.append(nf)
        return Meshes(verts=nl, faces=fl)
    def __call__(self, meshes, **kwargs): return self.forward(meshes, **kwargs)

def mesh_laplacian_smoothing(meshes, method="uniform"):
    return torch.tensor(0.0, device=meshes.device if hasattr(meshes, "device") else "cpu")
def mesh_normal_consistency(meshes):
    return torch.tensor(0.0, device=meshes.device if hasattr(meshes, "device") else "cpu")
"""

_w(f"{_PT3D}/_shim.py", _SHIM_SRC.strip())

# Write __init__.py files (simple approach avoids escaping hell)
with open(f"{_PT3D}/__init__.py", "w") as _f:
    _f.write("from ._shim import *\n")
    _f.write("from ._shim import __version__\n")
    _f.write("class _C:\n")
    _f.write("    def __getattr__(self, name):\n")
    _f.write('        raise ImportError(f"pytorch3d._C.{name} not available")\n')
    _f.write("C = _C()\n")
for _sub in ["io", "structures", "renderer", "ops", "loss", "common"]:
    with open(f"{_PT3D}/{_sub}/__init__.py", "w") as _f:
        _f.write("from pytorch3d._shim import *\n")
with open(f"{_PT3D}/renderer/mesh/__init__.py", "w") as _f:
    _f.write("from pytorch3d._shim import rasterize_meshes, MeshRasterizer\n")
    _f.write("from pytorch3d._shim import MeshRenderer, SoftPhongShader\n")
    _f.write("from pytorch3d._shim import SoftSilhouetteShader, RasterizationSettings\n")
    _f.write("from pytorch3d._shim import Fragments, TexturesUV, TexturesVertex\n")
    _f.write("from pytorch3d._shim import TexturesAtlas, BlendParams, PointLights\n")
    _f.write("from pytorch3d._shim import PerspectiveCameras, AmbientLights\n")
os.makedirs(f"{_PT3D}/renderer/blending", exist_ok=True)
with open(f"{_PT3D}/renderer/blending/__init__.py", "w") as _f:
    _f.write("from pytorch3d._shim import *\n")
    _f.write("from pytorch3d._shim import softmax_rgb_blend, sigmoid_alpha_blend, hard_rgb_blend\n")
open(f"{_PT3D}/tools/__init__.py", "w").close()

# Import shim classes into notebook namespace
_sys.path.insert(0, _os.path.expanduser("~"))
from pytorch3d._shim import Meshes, Pointclouds, load_obj, save_obj, load_objs_as_meshes
from pytorch3d._shim import MeshRenderer, MeshRasterizer, RasterizationSettings, Fragments
from pytorch3d._shim import SoftPhongShader, SoftSilhouetteShader, PointLights, PerspectiveCameras
from pytorch3d._shim import TexturesVertex, TexturesUV, TexturesAtlas, BlendParams, AmbientLights
from pytorch3d._shim import SubdivideMeshes, rasterize_meshes, softmax_rgb_blend, sigmoid_alpha_blend
from pytorch3d._shim import list_to_padded, padded_to_list, list_to_packed, packed_to_list
from pytorch3d._shim import mesh_laplacian_smoothing, mesh_normal_consistency, knn_points
from pytorch3d._shim import packed_to_padded, padded_to_packed, make_device, TensorProperties
print(f"[pytorch3d] Phase 0-4 foundation ready at {_PT3D}/ (SubdivideMeshes: trimesh.subdivide)")
# ---- end shim ----
try:
    import pymeshlab
    print("pymeshlab: already installed")
except ImportError:
    print("pymeshlab: not found, installing...")
    import subprocess, sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "pymeshlab"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    import pymeshlab
    print("pymeshlab: installed")

# ---- Verify pymeshlab (fallback install if kernel restart broke it) ----
try:
    import pymeshlab
    print("pymeshlab: available")
except ImportError:
    print("pymeshlab: not found, (re)installing...")
    import subprocess, sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "pymeshlab"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    import pymeshlab
    print("pymeshlab: (re)installed successfully")

# Verify pymeshlab can load OBJ format
try:
    _test_ms = pymeshlab.MeshSet()
    import tempfile as _tf
    with _tf.NamedTemporaryFile(suffix='.obj', mode='w', delete=False) as _f:
        _f.write('v 0 0 0\nv 1 0 0\nv 0 1 0\nf 1 2 3\n')
        _test_path = _f.name
    _test_ms.load_new_mesh(_test_path)
    os.unlink(_test_path)
    _pymeshlab_works = True
    print("pymeshlab: OBJ format OK")
except Exception as _e:
    _pymeshlab_works = False
    print(f"pymeshlab: OBJ format FAILED ({_e}), will use trimesh fallback")

# ---- Generate missing garment templates (Robust) ----
import numpy as np
import pickle
from collections import deque

tmps_dir = GARMENT_REC_DIR / "data" / "tmps"
GARMENTS_META = [
    ("T-shirt", 1954), ("front_open_T-shirt", 1954),
    ("Shirt", 2468), ("front_open_Shirt", 2468),
    ("Shorts", 678), ("Pants", 1180),
]

for gtype, vnum in GARMENTS_META:
    try:
        gar_dir = tmps_dir / gtype
        obj_path = gar_dir / "garment_tmp.obj"
        dense_obj_path = gar_dir / "garment_tmp_subdivide_uv_new.obj"
        spiral_path = gar_dir / "spiral_indices_2.npy"
        pca_path = gar_dir / "pca_data_64.npz"
        tex_uv_path = gar_dir / "tex_uv.pkl"

        if obj_path.exists() and dense_obj_path.exists() and spiral_path.exists() and obj_path.stat().st_size > 0:
            print(f"{gtype}: templates cached")
            continue
            
        if not pca_path.exists() or not tex_uv_path.exists():
            print(f"*** {gtype}: SKIPPING, base assets missing (PCA/UV) ***")
            continue

        print(f"Processing {gtype}...")
        import numpy as _np; pca = _np.load(str(pca_path))
        verts = pca["mean"].reshape(-1, 3)
        tex_uv = pickle.load(open(str(tex_uv_path), "rb"))
        
        # Determine UV source (support both tensor and ndarray)
        fu = tex_uv["faces_uvs"]
        vu = tex_uv["verts_uvs"]
        faces_uvs = fu.numpy() if hasattr(fu, "numpy") else fu
        verts_uvs = vu.numpy() if hasattr(vu, "numpy") else vu

        # Write obj
        with open(str(obj_path), 'w') as f:
            for v in verts: f.write(f'v {v[0]:.8f} {v[1]:.8f} {v[2]:.8f}\n')
            for vt in verts_uvs: f.write(f'vt {vt[0]:.8f} {vt[1]:.8f}\n')
            for face in faces_uvs: f.write(f'f {face[0]+1}/{face[0]+1} {face[1]+1}/{face[1]+1} {face[2]+1}/{face[2]+1}\n')

        # Re-export .obj via trimesh to ensure valid format
        try:
            import trimesh as _tm_check
            _tm_mesh = _tm_check.load(str(obj_path), force='mesh')
            _tm_mesh.export(str(obj_path))
        except Exception:
            pass

        # Subdivision
        if _pymeshlab_works:
            try:
                import pymeshlab
                ms = pymeshlab.MeshSet()
                ms.load_new_mesh(str(obj_path))
                _Threshold = getattr(pymeshlab, 'PercentageValue', None) or getattr(pymeshlab, 'Percentage', None)
                ms.meshing_surface_subdivision_loop(loopweight=0, iterations=2, threshold=_Threshold(0) if _Threshold else 0)
                m = ms.current_mesh()
                dense_verts = m.vertex_matrix()
                dense_faces = m.face_matrix()
                with open(str(dense_obj_path), 'w') as f:
                    for v in dense_verts: f.write(f'v {v[0]:.8f} {v[1]:.8f} {v[2]:.8f}\n')
                    if m.has_vertex_tex_coord():
                        vt = m.vertex_tex_coord_matrix()
                        for uv in vt: f.write(f'vt {uv[0]:.8f} {uv[1]:.8f}\n')
                        for face in dense_faces:
                            f.write(f'f {face[0]+1}/{face[0]+1} {face[1]+1}/{face[1]+1} {face[2]+1}/{face[2]+1}\n')
                    else:
                        for face in dense_faces:
                            f.write(f'f {face[0]+1} {face[1]+1} {face[2]+1}\n')
            except Exception as _sub_e:
                print(f"  pymeshlab subdivision failed: {_sub_e}, copying base mesh as dense")
                import shutil as _sh
                _sh.copy2(str(obj_path), str(dense_obj_path))
        else:
            # Trimesh fallback for subdivision
            try:
                import trimesh as _tm
                _mesh = _tm.load(str(obj_path), force='mesh')
                for _ in range(2):
                    _mesh = _mesh.subdivide()
                _verts = _mesh.vertices
                _faces = _mesh.faces
                with open(str(dense_obj_path), 'w') as f:
                    for v in _verts: f.write(f'v {v[0]:.8f} {v[1]:.8f} {v[2]:.8f}\n')
                    for face in _faces: f.write(f'f {face[0]+1} {face[1]+1} {face[2]+1}\n')
                print(f"  {gtype}: trimesh subdivision OK ({len(_verts)} verts)")
            except Exception as _tm_e:
                print(f"  trimesh subdivision failed: {_tm_e}, copying base mesh")
                import shutil as _sh
                _sh.copy2(str(obj_path), str(dense_obj_path))
        # Spiral (using in-memory verts/faces_uvs instead of openmesh)
        V = verts.shape[0]
        faces_for_adj = np.array(faces_uvs, dtype=int)
        adj = {i: set() for i in range(V)}
        for face in faces_for_adj:
            for i in range(3):
                for j in range(i+1, 3):
                    v0, v1 = face[i], face[j]
                    adj[v0].add(v1); adj[v1].add(v0)
        max_neighbors = 20
        spiral = np.zeros((V, max_neighbors), dtype=np.int64)
        for v in range(V):
            seq = [v]; visited = {v}
            q = deque([(nbr, 1) for nbr in sorted(adj[v])])
            visited.update(adj[v]); seq.extend(sorted(adj[v]))
            while len(seq) < max_neighbors and q:
                cur, dist = q.popleft()
                for nbr in sorted(adj[cur]):
                    if nbr not in visited and len(seq) < max_neighbors:
                        visited.add(nbr); seq.append(nbr); q.append((nbr, dist + 1))
            while len(seq) < max_neighbors: seq.append(v)
            spiral[v] = seq[:max_neighbors]
        np.save(str(spiral_path), spiral[np.newaxis, :, :])
        print(f"{gtype}: Done (Spiral: {spiral.shape[0]}x{max_neighbors})")
    except Exception as e:
        print(f"*** {gtype}: FAILED -> {e} ***")

print("GarmentRec templates: processing finished")


# ---- GarmentGPT: Clone repo + download checkpoints ----
GARMENT_GPT_DIR = WD / "Garment-GPT"
if not (GARMENT_GPT_DIR / "main.py").exists():
    print("Cloning Garment-GPT repo...")
    !git clone --depth=1 https://github.com/ChimerAI-MMLab/Garment-GPT.git {GARMENT_GPT_DIR} 2>&1 || echo "WARN: GarmentGPT clone failed (may already exist)"
else:
    print("Garment-GPT repo: cached")

# LLaMA-Factory (required by GarmentGPT main.py imports)
LLAMA_FACTORY = WD / "LLaMA-Factory"
if not (LLAMA_FACTORY / "setup.py").exists():
    print("Cloning LLaMA-Factory...")
    !git clone --depth=1 https://github.com/hiyouga/LLaMA-Factory.git {LLAMA_FACTORY} 2>&1 || echo "WARN: LLaMA-Factory clone failed (may already exist)"
    !pip install -q -e {LLAMA_FACTORY} 2>&1 | tail -3 || echo "WARN: LLaMA-Factory editable install failed"
else:
    print("LLaMA-Factory: cached")

# vLLM shim (P100 sm_60 can't run real vLLM — uses transformers + int8 quantization)
VLLM_SHIM = WD / "vllm_shim.py"
VLLM_SHIM.write_text('''
"""vLLM compatibility layer using HuggingFace Transformers + int8 quantization."""
import sys, types, torch

_vllm_real = None
try:
    import vllm as _vllm_real
    _HasVLLM = True
    print(f"[vllm-shim] Real vllm imported: CUDA {torch.cuda.get_device_capability()}")
except Exception as e:
    _HasVLLM = False
    print(f"[vllm-shim] Real vllm not available: {e}")

class _HF_LLM:
    def __init__(self, model, **kwargs):
        self._model = None
        self._processor = None
        self._real = None
        if _vllm_real is not None:
            try:
                self._real = _vllm_real.LLM(model=model, **kwargs)
                print("[vllm-shim] Using REAL vLLM engine")
                return
            except Exception as e:
                print(f"[vllm-shim] Real vLLM init failed: {e}")
        from transformers import AutoProcessor, AutoConfig
        from transformers import BitsAndBytesConfig
        print(f"[vllm-shim] Loading model {model} with int8 quantization...")
        bnb_config = BitsAndBytesConfig(load_in_8bit=True, llm_int8_threshold=6.0)
        self._processor = AutoProcessor.from_pretrained(model, trust_remote_code=True)
        from transformers import AutoModelForVision2Seq
        self._model = AutoModelForVision2Seq.from_pretrained(
            model, quantization_config=bnb_config, device_map="auto",
            trust_remote_code=True, torch_dtype=torch.float16,
        )
        self._model.eval()
        print(f"[vllm-shim] Model loaded on {self._model.device}")

    def generate(self, inputs_list, sampling_params):
        if self._real is not None:
            return self._real.generate(inputs_list, sampling_params)
        results = []
        for inp in inputs_list:
            prompt = inp.get("prompt", "")
            images = inp.get("multi_modal_data", {}).get("image", [])
            if images:
                proc = self._processor(text=prompt, images=images[0], return_tensors="pt").to(self._model.device)
            else:
                proc = self._processor(text=prompt, return_tensors="pt").to(self._model.device)
            mt = getattr(sampling_params, "max_tokens", 4096)
            tp = getattr(sampling_params, "temperature", 0.1)
            with torch.no_grad():
                out = self._model.generate(**proc, max_new_tokens=min(mt, 4096),
                    do_sample=tp > 0, temperature=max(tp, 0.01), top_p=0.9)
            gen = self._processor.decode(out[0], skip_special_tokens=False)
            class _O:
                def __init__(self, text): self.text = text
            class _RO:
                def __init__(self, outputs): self.outputs = outputs
            results.append(_RO(outputs=[_O(text=gen)]))
        return results

class _HF_SamplingParams:
    def __init__(self, temperature=0.1, max_tokens=4096, skip_special_tokens=False, seed=42, stop_token_ids=None):
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.skip_special_tokens = skip_special_tokens
        self.seed = seed
        self.stop_token_ids = stop_token_ids or []

vllm_mod = types.ModuleType("vllm")
vllm_mod.LLM = _HF_LLM
vllm_mod.SamplingParams = _HF_SamplingParams
sys.modules["vllm"] = vllm_mod
print("[vllm-shim] Installed (real vllm:", _HasVLLM, ")")
''')
sys.path.insert(0, str(WD))
import vllm_shim
print("vLLM shim activated")

# GarmentGPT checkpoints (inference-only: skip optimizer states, download ~14GB)
CHECKPOINTS_DIR = GARMENT_GPT_DIR / "checkpoints"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

# Inference files only: 3 model shards (~14GB) + config/tokenizer + codec models (~642MB)
# SKIP: global_step12844/ (~80GB optimizer states), rng_state_*.pth, scheduler.pt, training_args.bin
gpt_files = [
    "vlm/checkpoint-12844/model-00001-of-00003.safetensors",  # 4.98 GB
    "vlm/checkpoint-12844/model-00002-of-00003.safetensors",  # 4.99 GB
    "vlm/checkpoint-12844/model-00003-of-00003.safetensors",  # 4.19 GB
    "vlm/checkpoint-12844/model.safetensors.index.json",
    "vlm/checkpoint-12844/config.json",
    "vlm/checkpoint-12844/generation_config.json",
    "vlm/checkpoint-12844/preprocessor_config.json",
    "vlm/checkpoint-12844/processor_config.json",
    "vlm/checkpoint-12844/tokenizer.json",
    "vlm/checkpoint-12844/tokenizer.model",
    "vlm/checkpoint-12844/tokenizer_config.json",
    "vlm/checkpoint-12844/special_tokens_map.json",
    "vlm/checkpoint-12844/added_tokens.json",
    "vlm/checkpoint-12844/chat_template.json",
    "vqvae/best_model.pth",     # 622 MB (edge codec)
    "vqvae/rt_vqvae.pth",       # 20 MB (RT codec)
]

from concurrent.futures import ThreadPoolExecutor, as_completed

HF_AUTH = {"Authorization": f"Bearer {os.environ['HF_TOKEN']}"}
GPT_BASE = "https://huggingface.co/ChimerAI/GarmentGPT/resolve/main"

# Separate large safetensors (parallel) from small config/codec files (sequential)
large_files = [f for f in gpt_files if f.endswith(".safetensors") and "model-" in f]
small_files = [f for f in gpt_files if f not in large_files]

def download_gpt_file(f):
    dest = CHECKPOINTS_DIR / f
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        return True
    url = f"{GPT_BASE}/{f}?download=1"
    if robust_download(url, dest, extra_headers=HF_AUTH, desc=f):
        return True
    # Layer 2: hf_hub_download (different CDN path, may bypass Xet 403)
    print(f"  {f}: fallback to hf_hub_download...")
    try:
        hf_hub_download(
            repo_id="ChimerAI/GarmentGPT",
            filename=f,
            local_dir=str(CHECKPOINTS_DIR),
            local_dir_use_symlinks=False,
            force_download=True,
        )
        if dest.exists() and dest.stat().st_size > 0:
            return True
    except Exception as _e2:
        print(f"  {f}: hf_hub_download fallback failed: {_e2}")
    return False

need_download = not (CHECKPOINTS_DIR / "vqvae" / "best_model.pth").exists()
if need_download:
    # Parallel: 3 large safetensors (4.98 + 4.99 + 4.19 = ~14.2 GB, ~3x speedup)
    print("Downloading 3 large model shards in parallel...")
    with ThreadPoolExecutor(max_workers=3) as pool:
        fut_to_file = {pool.submit(download_gpt_file, f): f for f in large_files}
        for fut in as_completed(fut_to_file):
            f = fut_to_file[fut]
            if fut.result():
                size_mb = (CHECKPOINTS_DIR / f).stat().st_size / (1024*1024)
                print(f"  {f}: {size_mb:.0f} MB")
            else:
                print(f"  *** {f}: FAILED ***")
    
    # Sequential: smaller config/tokenizer/codec files
    for f in small_files:
        download_gpt_file(f)
    
    print("GarmentGPT checkpoints: done (~14.6GB model + 642MB codecs)")
else:
    print("GarmentGPT checkpoints: cached")

# ---- Kaggle Dataset auto-upload (cache for future runs) ----
try:
    _kaggle_dir = Path("/tmp/kaggle_dataset_upload")
    if _kaggle_dir.exists(): shutil.rmtree(_kaggle_dir)
    _kaggle_dir.mkdir(parents=True)
    _upload_files = {
        model_pth: "mrf_0.1_shading_0.1_pca64_ep100_bth0.pth",
        smpl_path: "neutral_smpl_with_cocoplus_reg.txt",
    }
    for src, name in _upload_files.items():
        if src.exists():
            (Path(_kaggle_dir) / name).symlink_to(src)
    import json as _json
    with open(_kaggle_dir / "dataset-metadata.json", "w") as _f:
        _json.dump({
            "title": "Korra Garment Weights",
            "id": "jacobthankgod/korra-garment-weights",
            "description": "Pretrained weights for GarmentRec + SMPL model, auto-uploaded from Kaggle notebook.",
            "licenses": [{"name": "MIT"}],
        }, _f)
    import subprocess as _sp
    _r = _sp.run(["kaggle", "datasets", "create", "-p", str(_kaggle_dir), "--dir-mode", "tar", "--quiet"],
                capture_output=True, text=True, timeout=120)
    if _r.returncode == 0:
        print(f"Kaggle Dataset '{KAGGLE_DATASET}': uploaded for future runs")
    elif "already exists" in (_r.stderr or ""):
        _r2 = _sp.run(["kaggle", "datasets", "version", "-p", str(_kaggle_dir), "--dir-mode", "tar", "--quiet"],
                    capture_output=True, text=True, timeout=120)
        if _r2.returncode == 0:
            print(f"Kaggle Dataset '{KAGGLE_DATASET}': version updated")
        else:
            print(f"Kaggle Dataset version update failed (non-fatal): {_r2.stderr[:200]}")
    else:
        print(f"Kaggle Dataset upload failed (non-fatal): {_r.stderr[:200]}")
except Exception as _e:
    print(f"Kaggle Dataset auto-upload skipped (non-fatal): {_e}")

# ---- SAM2 weights ----
sam2_dir = WEIGHTS_DIR / "sam2"
sam2_dir.mkdir(exist_ok=True)
if not (sam2_dir / "sam2_hiera_large.pt").exists():
    print("Downloading SAM2 weights...")
    !wget -q -O {sam2_dir}/sam2_hiera_large.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
else:
    print("SAM2: cached")

# ---- GarmentCode ----
garmentcode_dir = WEIGHTS_DIR / "garmentcode"
if not (garmentcode_dir / "repo").exists():
    print("Cloning GarmentCode...")
    !git clone --depth=1 https://github.com/maria-korosteleva/GarmentCode.git {garmentcode_dir}/repo
else:
    print("GarmentCode: cached")

print("\nStorage used:")
!du -sh {WEIGHTS_DIR}/* 2>/dev/null || true
!du -sh {GARMENT_REC_DIR} 2>/dev/null || true
!du -sh {GARMENT_GPT_DIR} 2>/dev/null || true











In [ ]:
import os
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN", "hf_YOUR_TOKEN_HERE"))
print("Logged in")


In [ ]:
%%writefile /root/api_server.py
"""
Garment Reconstruction API Server (Async + Split Pipeline + SAM2 Fix)
Runs on Kaggle GPU backend, exposed via Cloudflare Tunnel.

Pipeline: Image -> 3D Mesh + Sewing Pattern

Endpoints:
  POST /api/v1/reconstruct     -> async job (returns job_id, processes in background)
  POST /api/v1/segment         -> Step 1: rembg + SAM2 (~20s)
  POST /api/v1/mesh            -> Step 2: GarmentRec (~50s)
  POST /api/v1/pattern         -> Step 3: GarmentGPT (~50s)
  POST /api/v1/callback        -> receives result URL from processing
  GET  /api/v1/job/{job_id}    -> poll job status
  GET  /health                 -> health check
  GET  /debug/error            -> last error traceback
"""

# ---- vLLM compatibility layer (SHIM) ----
import sys, types, torch

def _activate_vllm_shim():
    import importlib
    _orig_find_spec = importlib.util.find_spec
    def _patched_find_spec(name, *args, **kwargs):
        if name == 'vllm':
            return None
        return _orig_find_spec(name, *args, **kwargs)
    importlib.util.find_spec = _patched_find_spec

    class _HF_LLM:
        def __init__(self, model, **kwargs):
            from transformers import AutoProcessor, AutoConfig
            import torch

            use_8bit = False
            gpu_label = "cpu"
            if torch.cuda.is_available():
                cap = torch.cuda.get_device_capability()
                gpu_label = f"sm_{cap[0]}{cap[1]}"
                if cap >= (7, 0):
                    use_8bit = True

            print(f'[vllm-shim] Loading model {model} on {gpu_label}...')
            self._processor = AutoProcessor.from_pretrained(model, trust_remote_code=True)
            cfg_kwargs = dict(kwargs)
            cfg_kwargs.setdefault('trust_remote_code', True)
            cfg = AutoConfig.from_pretrained(model, **cfg_kwargs)
            if hasattr(cfg, 'vision_config'):
                try:
                    from transformers import AutoModelForVision2Seq
                    model_class = AutoModelForVision2Seq
                except ImportError:
                    from transformers import LlavaForConditionalGeneration
                    model_class = LlavaForConditionalGeneration
            else:
                from transformers import AutoModelForCausalLM
                model_class = AutoModelForCausalLM

            load_kwargs = dict(
                device_map='auto', trust_remote_code=True, torch_dtype=torch.float16,
            )
            if use_8bit:
                from transformers import BitsAndBytesConfig
                load_kwargs['quantization_config'] = BitsAndBytesConfig(
                    load_in_8bit=True, llm_int8_threshold=6.0,
                )

            self._model = model_class.from_pretrained(model, **load_kwargs)
            self._model.eval()
            dtype_label = "int8" if use_8bit else "fp16"
            print(f'[vllm-shim] Model loaded on {self._model.device} in {dtype_label}')

        def generate(self, inputs_list, sampling_params):
            results = []
            for inp in inputs_list:
                prompt = inp.get('prompt', '')
                images = inp.get('multi_modal_data', {}).get('image', [])
                if images:
                    proc = self._processor(text=prompt, images=images[0], return_tensors='pt').to(self._model.device)
                else:
                    proc = self._processor(text=prompt, return_tensors='pt').to(self._model.device)
                mt = getattr(sampling_params, 'max_tokens', 4096)
                tp = getattr(sampling_params, 'temperature', 0.1)
                with torch.no_grad():
                    out = self._model.generate(**proc, max_new_tokens=min(mt, 4096),
                        do_sample=tp > 0, temperature=max(tp, 0.01), top_p=0.9)
                gen = self._processor.decode(out[0], skip_special_tokens=False)
                class _O:
                    def __init__(self, text): self.text = text
                class _RO:
                    def __init__(self, outputs): self.outputs = outputs
                results.append(_RO(outputs=[_O(text=gen)]))
            return results

    class _HF_SamplingParams:
        def __init__(self, temperature=0.1, max_tokens=4096, skip_special_tokens=False, seed=42, stop_token_ids=None):
            self.temperature = temperature
            self.max_tokens = max_tokens
            self.skip_special_tokens = skip_special_tokens
            self.seed = seed
            self.stop_token_ids = stop_token_ids or []

    vllm_mod = types.ModuleType('vllm')
    vllm_mod.LLM = _HF_LLM
    vllm_mod.SamplingParams = _HF_SamplingParams
    sys.modules['vllm'] = vllm_mod
    print('[vllm-shim] vLLM compatibility shim installed (transformers + int8 backend)')

_activate_vllm_shim()

import os
import io
import json
import time
import uuid
import zipfile
import tempfile
import logging
import gc
import asyncio
import threading
import traceback
from pathlib import Path
from datetime import datetime
from contextlib import asynccontextmanager

import numpy as np
import torch
from PIL import Image

# Ensure python-multipart is available (torch --force-reinstall can wipe it)
import subprocess as _sp, sys as _s
try:
    import multipart  # noqa: F401
except ImportError:
    _sp.check_call([_s.executable, "-m", "pip", "install", "-q", "python-multipart"])

from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import StreamingResponse, JSONResponse
import uvicorn

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("garment-api")

def _resolve_working_dir() -> Path:
    env_dir = os.getenv("GARMENT_WORKING_DIR")
    if env_dir:
        try:
            candidate = Path(env_dir).expanduser()
            candidate.mkdir(parents=True, exist_ok=True)
            return candidate.resolve()
        except OSError:
            pass

    # Modal: /root is the working dir; Kaggle: /kaggle/working
    for candidate in [Path("/root"), Path("/kaggle/working"), Path.cwd()]:
        try:
            if not candidate.exists():
                continue  # Don't CREATE dirs — only use existing ones
            return candidate.resolve()
        except OSError:
            continue

    # Fallback: create /root if nothing else exists
    fallback = Path("/root")
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback.resolve()


WORKING_DIR = _resolve_working_dir()
WEIGHTS_DIR = WORKING_DIR / "weights"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_DEVICE = DEVICE
CPU_DEVICE = "cpu"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# ---- repo paths (cloned by Cell 2) ----
GARMENT_REC_DIR = WORKING_DIR / "GarmentRec"
GARMENT_GPT_DIR = WORKING_DIR / "Garment-GPT"
sys.path.insert(0, str(GARMENT_REC_DIR / "code"))
sys.path.insert(0, str(GARMENT_REC_DIR))
sys.path.insert(0, str(GARMENT_GPT_DIR))

SAM2_CKPT = WEIGHTS_DIR / "sam2" / "sam2_hiera_large.pt"

# ---- EC2 proxy callback URL ----
EC2_CALLBACK_URL = os.getenv("EC2_CALLBACK_URL", "https://korra.work/api/v2/garment/callback")

# ---- globals (populated by load_models) ----
rembg_remove = None
predict_fn = None
sam2_model = None
garmentrec_model = None
garmentgpt_predictor = None

# ---- async model loading state ----
models_loaded = threading.Event()
loading_state = {}
loading_errors = {}

# ---- async job queue ----
jobs = {}
job_progress = {}  # {job_id: {"stage": str, "progress": int, "message": str, "sequence": int}}
job_progress_events = {}  # {job_id: [asyncio.Event]} — for SSE subscribers

# ---- health tracking ----
SERVER_START_TIME = time.time()
REQUEST_COUNT = 0
ERROR_COUNT = 0
MODEL_LOAD_TIMES = {}

# Phase 209: GPU cost tracking
gpu_usage_log = {}  # {user_id: {"total_gpu_sec": float, "jobs": int, "last_job": str}}
COST_PER_GPU_MINUTE_USD = 0.50  # T4 on-demand estimate (~$0.35-0.65)

# ---- structured error codes (Phase 31) ----
ERROR_CODES = {
    "CUDA_OOM": {"code": "CUDA_OOM", "message": "GPU out of memory", "recoverable": True},
    "SAM2_LOAD_FAILED": {"code": "SAM2_LOAD_FAILED", "message": "SAM2 model failed to load", "recoverable": True},
    "GARMENTREC_LOAD_FAILED": {"code": "GARMENTREC_LOAD_FAILED", "message": "GarmentRec model failed to load", "recoverable": True},
    "GARMENTGPT_LOAD_FAILED": {"code": "GARMENTGPT_LOAD_FAILED", "message": "GarmentGPT model failed to load", "recoverable": True},
    "REMBG_FAILED": {"code": "REMBG_FAILED", "message": "Background removal failed", "recoverable": True},
    "MODEL_CORRUPT": {"code": "MODEL_CORRUPT", "message": "Model checkpoint is corrupt or incompatible", "recoverable": False},
    "INFERENCE_FAILED": {"code": "INFERENCE_FAILED", "message": "Model inference failed", "recoverable": True},
    "TIMEOUT": {"code": "TIMEOUT", "message": "Request timed out", "recoverable": True},
    "INVALID_IMAGE": {"code": "INVALID_IMAGE", "message": "Invalid or unreadable image", "recoverable": False},
    "IMAGE_TOO_LARGE": {"code": "IMAGE_TOO_LARGE", "message": "Image exceeds size limit", "recoverable": False},
    "UNKNOWN": {"code": "UNKNOWN", "message": "An unexpected error occurred", "recoverable": False},
}


def _structured_error(err_key, detail=None, extra=None):
    """Build a structured error response dict."""
    entry = ERROR_CODES.get(err_key, ERROR_CODES["UNKNOWN"])
    out = {"error": dict(entry)}
    if detail:
        out["error"]["detail"] = str(detail)
    if extra:
        out["error"]["extra"] = extra
    return out


def _write_error_context(error_key, detail=None, request_path=None):
    """Write rich error context to /kaggle/working/last_error.txt (Phase 33)."""
    try:
        import datetime
        parts = [
            f"=== Error at {datetime.datetime.utcnow().isoformat()}Z ===",
            f"Code: {error_key}",
            f"Detail: {detail}" if detail else "",
            f"Path: {request_path}" if request_path else "",
            f"GPU allocated: {torch.cuda.memory_allocated() / (1024**3):.2f}GB" if torch.cuda.is_available() else "GPU: N/A",
            f"GPU reserved: {torch.cuda.memory_reserved() / (1024**3):.2f}GB" if torch.cuda.is_available() else "",
            f"Models: rembg={rembg_remove is not None} sam2={sam2_model is not None} " +
            f"garmentrec={garmentrec_model is not None} garmentgpt={garmentgpt_predictor is not None}",
            f"Loading state: {loading_state}",
            "=" * 60,
        ]
        (WORKING_DIR / "last_error.txt").write_text("\n".join(parts))
    except Exception:
        pass


# ---- retry logic (Phase 32) ----
def _retry_on_oom(func, max_retries=2):
    """Retry a model load if it fails with CUDA OOM, clearing cache between attempts."""
    for attempt in range(max_retries + 1):
        try:
            return func()
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and attempt < max_retries:
                logger.warning(f"CUDA OOM on attempt {attempt+1}/{max_retries+1}, clearing cache and retrying...")
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue
            raise


# Patch PIL._typing for SAM2 compatibility (Pillow 10+ removed _Ink)
try:
    import PIL._typing as _pil_typing
    if not hasattr(_pil_typing, '_Ink'):
        _pil_typing._Ink = type(None)
except ImportError:
    pass

# ═══════════════════════════════════════════════════════════════
#  SAM2 segmentation (with checkpoint compatibility patch)
# ═══════════════════════════════════════════════════════════════

def _load_sam2():
    """Load SAM2 on CPU. Handles checkpoint version mismatches by filtering unexpected keys.
    Disables CUDA cache warmup to prevent OOM on restart (leaked GPU memory from previous process)."""
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor

    ckpt = str(SAM2_CKPT)
    if not Path(ckpt).exists():
        raise FileNotFoundError(f"SAM2 checkpoint not found at {ckpt}")

    # Patch build_sam2 to filter mismatched keys (newer sam2 package vs older checkpoint)
    import sam2.build_sam as _build_mod
    _orig_load_checkpoint = _build_mod._load_checkpoint

    def _patched_load_checkpoint(model, ckpt_path):
        sd = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        if "model" in sd:
            sd = sd["model"]
        model_keys = set(model.state_dict().keys())
        ckpt_keys = set(sd.keys())
        unexpected = ckpt_keys - model_keys
        if unexpected:
            logger.warning(f"SAM2: filtering {len(unexpected)} unexpected keys from checkpoint: {unexpected}")
        filtered = {k: v for k, v in sd.items() if k in model_keys}
        model.load_state_dict(filtered, strict=False)

    _build_mod._load_checkpoint = _patched_load_checkpoint
    try:
        # Disable SAM2's CUDA cache warmup to prevent OOM on server restart.
        # The warmup hardcodes torch.device("cuda") in PositionEmbeddingSine.__init__,
        # bypassing the torch.device("cpu") context. Hydra overrides disable it.
        hydra_overrides = [
            "++model.image_encoder.neck.position_encoding.warmup_cache=false",
            "++model.memory_encoder.position_encoding.warmup_cache=false",
        ]
        with torch.device("cpu"):
            m = build_sam2("sam2_hiera_l.yaml", ckpt, device=CPU_DEVICE,
                           hydra_overrides_extra=hydra_overrides)
    finally:
        _build_mod._load_checkpoint = _orig_load_checkpoint

    pred = SAM2ImagePredictor(m)
    torch.cuda.empty_cache()
    gc.collect()
    logger.info("SAM2 loaded on CPU")
    return m, pred


def segment_garment(image: Image.Image) -> np.ndarray:
    """Real SAM2-based garment segmentation. Returns a binary mask (H,W) uint8 0/255."""
    global predict_fn
    img_np = np.array(image.convert("RGB"))
    h, w = img_np.shape[:2]
    predict_fn.set_image(img_np)
    masks, scores, _ = predict_fn.predict(
        point_coords=np.array([[w // 2, h // 2]]),
        point_labels=np.array([1]),
        multimask_output=True,
    )
    best = masks[scores.argmax()]
    return (best.astype(np.uint8) * 255)


# ═══════════════════════════════════════════════════════════════
#  GarmentRec — 3D mesh from a single image
# ═══════════════════════════════════════════════════════════════

def _load_garmentrec():
    """Load GarmentRec model on CPU. Fails if assets or SMPL model are missing."""
    import pymeshlab as _pml
    _Percentage = getattr(_pml, 'PercentageValue', None) or getattr(_pml, 'Percentage', None)
    if _Percentage is not None:
        _pml.Percentage = _Percentage
    from module.ImageReconstructModel import ImageReconstructModel
    from module.SkinWeightModel import SkinWeightNet
    import pickle

    gar_dir = GARMENT_REC_DIR / "code"
    smpl_path = GARMENT_REC_DIR / "smpl_pytorch" / "model" / "neutral_smpl_with_cocoplus_reg.txt"
    if not smpl_path.exists():
        raise FileNotFoundError(
            f"SMPL model not found at {smpl_path}. "
            "Download from https://smpl.is.tue.mpg.de/ and place the file there."
        )
    midpair_path = gar_dir.parent / "data" / "midpairs.pkl"
    dense_midpair_path = gar_dir.parent / "data" / "dense_midpairs.pkl"
    pca_folder = gar_dir.parent / "data" / "tmps"
    dense_template_folder = gar_dir.parent / "data"
    model_path = gar_dir.parent / "models" / "mrf_0.1_shading_0.1" / "mrf_0.1_shading_0.1_pca64_ep100_bth0.pth"

    for p in [midpair_path, dense_midpair_path, pca_folder, model_path]:
        if not p.exists():
            raise FileNotFoundError(f"GarmentRec asset not found: {p}")

    with open(midpair_path, "rb") as f:
        midpairs = pickle.load(f)
    with open(dense_midpair_path, "rb") as f:
        dense_midpairs = pickle.load(f)

    garments = ["T-shirt", "front_open_T-shirt", "Shirt", "front_open_Shirt", "Shorts", "Pants"]
    garmentvnums = [1954, 1954, 2468, 2468, 678, 1180]
    upper_type_num = 4
    pca_dim = 64
    tran_mean = [0.0, 0.0, 0.0]

    skin_net = SkinWeightNet(4, True)
    device = CPU_DEVICE

    net = ImageReconstructModel(
        skin_net,
        with_classification=True,
        tran_mean=tran_mean,
        garments=garments,
        garmentvnums=garmentvnums,
        upper_type_num=upper_type_num,
        pca_folder=str(pca_folder),
        pca_dim=pca_dim,
        smpl_model_path=str(smpl_path),
        midpairs=midpairs,
        infer_camera=True,
        infer_tex=True,
        inferring=True,
        use_detail=True,
        mesh_save_folder=None,
        vis_save_folder=None,
        dense_template_folder=str(dense_template_folder),
        displacement_scale=0.005,
        upsample_dismap=False,
        use_neighbor=True,
        device=device,
    )
    try:
        state = torch.load(str(model_path), map_location=device, weights_only=False)
    except (EOFError, pickle.UnpicklingError, RuntimeError) as _e:
        logger.error(f"Corrupt GarmentRec weights at {model_path}: {_e}")
        logger.error("Attempting re-download via hf_hub_download...")
        from huggingface_hub import hf_hub_download
        model_path.unlink(missing_ok=True)
        try:
            _new = hf_hub_download(
                repo_id='jacobthankgod4/smpl-model-garmentrec',
                filename='mrf_0.1_shading_0.1_pca64_ep100_bth0.pth',
                local_dir=str(model_path.parent),
                local_dir_use_symlinks=False,
            )
            state = torch.load(str(_new), map_location=device, weights_only=False)
            logger.info(f"Re-downloaded OK ({Path(_new).stat().st_size/(1024*1024):.0f} MB)")
        except Exception as _e2:
            raise RuntimeError(
                f"GarmentRec weights corrupt and re-download failed: {_e2}"
            ) from _e
    # Fix checkpoint mismatches: transpose SMPL matrices, skip incompatible keys
    model_state = net.state_dict()
    fixed_state = {}
    skipped = []
    for key in state:
        if key not in model_state:
            skipped.append(key)
            continue
        ckpt_shape = state[key].shape
        model_shape = model_state[key].shape
        if ckpt_shape == model_shape:
            fixed_state[key] = state[key]
        elif len(ckpt_shape) == 2 and len(model_shape) == 2 and ckpt_shape[0] == model_shape[1] and ckpt_shape[1] == model_shape[0]:
            logger.warning("Transposing %s: %s -> %s", key, ckpt_shape, model_shape)
            fixed_state[key] = state[key].T.contiguous()
        else:
            logger.warning("Skipping %s: ckpt %s != model %s", key, ckpt_shape, model_shape)
            skipped.append(key)
    if skipped:
        logger.info("Skipped %d mismatched keys in state_dict", len(skipped))
    net.load_state_dict(fixed_state, strict=False)

    # Fix SMPL J_regressor orientation: standard SMPL stores (24, 6890) but
    # GarmentRec's SMPL.py expects (6890, 24) for matmul(v_shaped, J_regressor).
    if hasattr(net.smpl, 'J_regressor') and net.smpl.J_regressor.shape == (24, 6890):
        net.smpl.J_regressor.data = net.smpl.J_regressor.T.contiguous()
        logger.info("Patched SMPL J_regressor: transposed (24,6890) -> (6890,24)")

    net = net.to(device)
    net.eval()
    logger.info("GarmentRec loaded on CPU")
    return net


def run_garmentrec(net, image: Image.Image, temp_dir: str) -> dict:
    """Run GarmentRec inference, save mesh to temp_dir, return mesh data."""
    import cv2
    import trimesh

    net.mesh_save_folder = temp_dir

    img_np = np.array(image.convert("RGB"))
    img_np = cv2.resize(img_np, (540, 540))
    img_np = img_np.astype(np.float32) / 255.0
    img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).to(CPU_DEVICE)
    names = np.array(["input.png"])

    cam_k = torch.Tensor(
        [[3.0375e03, 0.0, 270.0], [0.0, 3.0375e03, 270.0], [0.0, 0.0, 1.0]]
    ).to(CPU_DEVICE)
    bcnet_tran_mean = [-0.010962, 0.28778, 12.973]
    tran_mean = [0.0, 0.0, 0.0]
    cam_Rt = torch.tensor(
        [
            [1, 0, 0, tran_mean[0] - bcnet_tran_mean[0]],
            [0, 1, 0, tran_mean[1] - bcnet_tran_mean[1]],
            [0, 0, 1, tran_mean[2] - bcnet_tran_mean[2]],
        ]
    )
    cam_k = cam_k.matmul(cam_Rt).to(CPU_DEVICE)

    imgs_perg = torch.cat((img_tensor, img_tensor), 1).reshape(-1, 3, 540, 540)
    input_gtypes = np.array([[-1, -1]])

    with torch.no_grad():
        up_prob, bottom_prob, cam_Rs, cam_Ts, dis_maps = net(
            img_tensor, names, gtypes=input_gtypes, cam_k=cam_k, imgs_perg=imgs_perg
        )

    up_idx = up_prob.argmax(dim=1).item()
    bottom_idx = bottom_prob.argmax(dim=1).item()
    up_name = ["T-shirt", "front_open_T-shirt", "Shirt", "front_open_Shirt"][up_idx]
    bi = bottom_idx - 4 if bottom_idx >= 4 else bottom_idx
    bottom_name = ["Shorts", "Pants"][bi]
    logger.info(f"GarmentRec classified: upper={up_name}, lower={bottom_name}")

    up_path = os.path.join(temp_dir, "input_up.obj")
    bottom_path = os.path.join(temp_dir, "input_bottom.obj")

    mesh_data = {"upper": None, "lower": None}
    if os.path.exists(up_path):
        mesh = trimesh.load(up_path)
        v = mesh.vertices
        f = mesh.faces
        mesh_data["upper"] = {
            "vertices": v.tolist(),
            "faces": f.tolist(),
            "type": up_name,
        }
    if os.path.exists(bottom_path):
        mesh = trimesh.load(bottom_path)
        v = mesh.vertices
        f = mesh.faces
        mesh_data["lower"] = {
            "vertices": v.tolist(),
            "faces": f.tolist(),
            "type": bottom_name,
        }
    return mesh_data


# ═══════════════════════════════════════════════════════════════
#  GarmentGPT — sewing pattern from a single image
# ═══════════════════════════════════════════════════════════════

def _load_garmentgpt():
    """Load GarmentGPT predictor. LLM on GPU, codecs on CPU."""
    from main import GarmentPredictor

    gpt_dir = GARMENT_GPT_DIR
    llm_path = str(gpt_dir / "checkpoints" / "vlm" / "checkpoint-12844")
    codec_cfg = str(gpt_dir / "configs" / "config_vq1024_resres_aug_decay0.99_q5_gcd_nl8_ld512.yaml")
    rt_cfg = str(gpt_dir / "configs" / "config_rt_euler.yaml")

    for p in [llm_path, codec_cfg, rt_cfg]:
        if not Path(p).exists():
            raise FileNotFoundError(f"GarmentGPT asset not found: {p}")

    _orig_cwd = os.getcwd()
    os.chdir(str(gpt_dir))
    try:
        predictor = GarmentPredictor(
            llm_model_path=llm_path,
            codec_config_path=codec_cfg,
            rt_config_path=rt_cfg,
            device=GPU_DEVICE,
        )
    finally:
        os.chdir(_orig_cwd)

    predictor.codec_model = predictor.codec_model.to(CPU_DEVICE)
    predictor.rt_model = predictor.rt_model.to(CPU_DEVICE)
    torch.cuda.empty_cache()
    logger.info("GarmentGPT loaded (LLM on GPU, codecs on CPU)")
    return predictor


def run_garmentgpt(predictor, image: Image.Image) -> dict:
    """Save image to a temp file, run GarmentGPT predict, return GCD JSON."""
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
        tmp_path = f.name
        image.save(tmp_path, "PNG")
    try:
        result = predictor.predict(image_path=tmp_path)
        if result is None:
            raise RuntimeError("GarmentGPT prediction returned None")
        return result
    finally:
        try:
            os.unlink(tmp_path)
        except Exception:
            pass


# ═══════════════════════════════════════════════════════════════
#  rembg (background removal) — with PIL fallback
# ═══════════════════════════════════════════════════════════════

def _simple_remove_background(image, threshold=240):
    """PIL-based background removal fallback when rembg is unavailable."""
    import numpy as np
    arr = np.array(image.convert("RGBA"), dtype=np.uint8)
    bg = np.all(arr[:,:,:3] > threshold, axis=2)
    arr[bg, 3] = 0
    result = Image.fromarray(arr, mode="RGBA")
    canvas = Image.new("RGB", result.size, (255, 255, 255))
    canvas.paste(result, mask=result.split()[3])
    return canvas


def _load_rembg():
    """Robust rembg loader with PIL fallback. Handles numpy 2.x / scipy incompat."""
    global rembg_remove
    import subprocess, sys as _sys

    # Step 1: Ensure onnxruntime
    try:
        import onnxruntime
    except ImportError:
        logger.warning("onnxruntime not found, installing...")
        try:
            subprocess.check_call(
                [_sys.executable, "-m", "pip", "install", "onnxruntime"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except subprocess.CalledProcessError:
            logger.warning("onnxruntime install failed, using PIL fallback")
            rembg_remove = _simple_remove_background
            return

    # Step 2: Upgrade scipy for numpy 2.x compat (rembg pulls scipy via pymatting)
    try:
        import scipy, packaging.version as _v
        if _v.Version(scipy.__version__) < _v.Version("1.14.0"):
            raise ImportError
    except (ImportError, ValueError, AttributeError):
        logger.warning("scipy too old for numpy 2.x, upgrading...")
        try:
            subprocess.check_call(
                [_sys.executable, "-m", "pip", "install", "--upgrade", "scipy>=1.14.1"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except subprocess.CalledProcessError:
            pass

    # Step 3: Import rembg (retry with reinstall on failure)
    for attempt in range(2):
        try:
            from rembg import remove
            break
        except BaseException as _err:
            logger.warning(f"rembg import error (attempt {attempt+1}): {_err}")
            if attempt == 0:
                logger.warning("rembg import failed, reinstalling rembg --no-deps...")
                try:
                    subprocess.check_call(
                        [_sys.executable, "-m", "pip", "install", "--force-reinstall",
                         "rembg", "--no-deps"],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                    )
                    subprocess.check_call(
                        [_sys.executable, "-m", "pip", "install",
                         "onnxruntime", "Pillow", "numpy", "scipy>=1.14.1"],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                    )
                except subprocess.CalledProcessError as _e:
                    logger.warning(f"rembg reinstall failed: {_e}")
            else:
                logger.warning("rembg import failed after reinstall, using PIL fallback")
                rembg_remove = _simple_remove_background
                return

    # Step 4: Warm up (downloads u2net ~176MB)
    try:
        dummy = Image.new("RGB", (8, 8), (255, 255, 255))
        remove(dummy)
        rembg_remove = remove
        logger.info("rembg + u2net ready")
    except Exception as _warmup_err:
        logger.warning(f"rembg warmup failed ({_warmup_err}), using PIL fallback")
        # Try one more time with alpha_matting=False explicitly
        try:
            dummy = Image.new("RGB", (8, 8), (255, 255, 255))
            remove(dummy, alpha_matting_enabled=False)
            rembg_remove = remove
            logger.info("rembg ready (without alpha matting)")
        except Exception:
            logger.warning("rembg completely unavailable, using PIL fallback")
            rembg_remove = _simple_remove_background


# ═══════════════════════════════════════════════════════════════
#  Model loading
# ═══════════════════════════════════════════════════════════════

def _log_gpu_memory(tag: str = ""):
    """Log current GPU memory state for debugging CUDA OOM issues."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024**3)
        reserved = torch.cuda.memory_reserved() / (1024**3)
        logger.info(f"[GPU MEM] {tag} allocated={allocated:.2f}GB reserved={reserved:.2f}GB")


def _loading_thread():
    """Run model loading in a background thread so uvicorn can start immediately.
    Each model is loaded independently — one failure does NOT block others."""
    global rembg_remove, predict_fn, sam2_model, garmentrec_model, garmentgpt_predictor
    global loading_state, loading_errors, MODEL_LOAD_TIMES
    logger.info("Background model loading thread started")

    def _safe_load(name, load_fn, *args):
        """Load a model with individual error handling — never blocks other models."""
        global loading_errors
        t0 = time.time()
        try:
            loading_state[name] = "loading"
            result = load_fn(*args) if args else load_fn()
            loading_state[name] = "loaded"
            MODEL_LOAD_TIMES[name] = time.time() - t0
            logger.info(f"{name} loaded in {MODEL_LOAD_TIMES[name]:.1f}s")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return result
        except Exception as e:
            loading_state[name] = "failed"
            loading_errors[name] = traceback.format_exc()
            logger.error(f"{name} load failed: {e}")
            return None

    # rembg (critical — has PIL fallback built in)
    _safe_load("rembg", _load_rembg)

    # SAM2 (has fallback to simple foreground mask)
    result = _safe_load("sam2", lambda: _retry_on_oom(lambda: _load_sam2(), max_retries=2))
    if result is not None:
        sam2_model, predict_fn = result

    # GarmentRec (has trimesh shim fallback for pytorch3d)
    result = _safe_load("garmentrec", _load_garmentrec)
    if result is not None:
        garmentrec_model = result

    # GarmentGPT (has vllm shim fallback)
    result = _safe_load("garmentgpt", _load_garmentgpt)
    if result is not None:
        garmentgpt_predictor = result

    _log_gpu_memory("all_loaded")
    loaded = sum(1 for v in loading_state.values() if v == "loaded")
    logger.info(f"Model loading complete: {loaded}/4 models loaded")
    models_loaded.set()


# ═══════════════════════════════════════════════════════════════
#  Helper: build ZIP from pipeline results
# ═══════════════════════════════════════════════════════════════

def build_zip(image_id: str, mesh_data: dict = None, pattern_data: dict = None) -> bytes:
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as zf:
        if mesh_data:
            for part in ("upper", "lower"):
                md = mesh_data.get(part)
                if md:
                    obj_lines = [f"# Garment {part} ({md['type']})"]
                    for v in md["vertices"]:
                        obj_lines.append(f"v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}")
                    for fa in md["faces"]:
                        obj_lines.append(f"f {fa[0]+1} {fa[1]+1} {fa[2]+1}")
                    zf.writestr(f"{image_id}/mesh_{part}.obj", "\n".join(obj_lines))
        if pattern_data:
            zf.writestr(f"{image_id}/sewing_pattern.json", json.dumps(pattern_data, indent=2))
        meta = {
            "image_id": image_id,
            "garmentrec": mesh_data is not None,
            "garmentgpt": pattern_data is not None,
            "generated_at": datetime.utcnow().isoformat(),
        }
        zf.writestr(f"{image_id}/metadata.json", json.dumps(meta, indent=2))
    buffer.seek(0)
    return buffer.read()


# ═══════════════════════════════════════════════════════════════
#  Helper: post result to EC2 proxy callback
# ═══════════════════════════════════════════════════════════════

def post_result_to_proxy(job_id: str, result_zip: bytes):
    """POST ZIP to EC2 proxy callback endpoint (fire-and-forget)."""
    try:
        import httpx
        files = {"file": (f"{job_id}.zip", io.BytesIO(result_zip), "application/zip")}
        data = {"job_id": job_id}
        resp = httpx.post(EC2_CALLBACK_URL, files=files, data=data, timeout=30)
        logger.info(f"Posted result to proxy: {resp.status_code}")
    except Exception as e:
        logger.error(f"Failed to post result to proxy: {e}")


# ═══════════════════════════════════════════════════════════════
#  Background task: full pipeline
# ═══════════════════════════════════════════════════════════════

async def process_full_pipeline(job_id: str, image_bytes: bytes, include_mesh: bool, include_pattern: bool):
    """Runs in background. Builds ZIP, stores in jobs dict, posts to EC2 proxy."""
    seq = 0
    def emit_progress(stage, progress, message):
        nonlocal seq
        seq += 1
        job_progress[job_id] = {"stage": stage, "progress": progress, "message": message, "sequence": seq}
        for evt in job_progress_events.get(job_id, []):
            evt.set()

    try:
        emit_progress("uploading", 5, "Processing image...")
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

        emit_progress("segmenting", 15, "Removing background...")
        nobg = rembg_remove(image)

        emit_progress("segmenting", 30, "Segmenting garment...")
        mask = segment_garment(image)

        mesh_data = None
        pattern_data = None

        if include_mesh and garmentrec_model is not None:
            emit_progress("meshing", 40, "Reconstructing 3D mesh...")
            with tempfile.TemporaryDirectory() as tmp:
                mesh_data = run_garmentrec(garmentrec_model, nobg, tmp)
            gc.collect()

        if include_pattern and garmentgpt_predictor is not None:
            emit_progress("patterning", 65, "Generating sewing pattern...")
            pattern_data = run_garmentgpt(garmentgpt_predictor, nobg)

        emit_progress("zipping", 90, "Packaging results...")
        result_zip = build_zip(job_id, mesh_data, pattern_data)
        jobs[job_id]["status"] = "completed"
        jobs[job_id]["result_zip"] = result_zip

        emit_progress("complete", 100, "Done!")

        # Post to EC2 proxy for persistent storage
        post_result_to_proxy(job_id, result_zip)
        logger.info(f"Job {job_id} completed successfully")
    except Exception as e:
        import traceback as tb
        err = tb.format_exc()
        jobs[job_id]["status"] = "failed"
        jobs[job_id]["error"] = str(e)
        emit_progress("error", -1, str(e)[:200])
        try:
            (WORKING_DIR / "last_error.txt").write_text(err)
        except Exception:
            pass
        logger.error(f"Job {job_id} failed: {e}")


# ═══════════════════════════════════════════════════════════════
#  FastAPI app
# ═══════════════════════════════════════════════════════════════

@asynccontextmanager
async def lifespan(app):
    # Start model loading in background thread — uvicorn serves immediately
    thread = threading.Thread(target=_loading_thread, daemon=True)
    thread.start()
    yield
    # Graceful shutdown: release models and free GPU memory
    global rembg_remove, predict_fn, sam2_model, garmentrec_model, garmentgpt_predictor
    logger.info("Shutting down — releasing models...")
    rembg_remove = None
    predict_fn = None
    sam2_model = None
    garmentrec_model = None
    garmentgpt_predictor = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    logger.info("Shutdown complete")


app = FastAPI(title="Garment Reconstruction API", lifespan=lifespan)


@app.middleware("http")
async def catch_all_errors(request, call_next):
    global REQUEST_COUNT, ERROR_COUNT
    path = request.url.path
    include_in_count = not path.startswith("/health") and not path.startswith("/ready")
    if include_in_count:
        REQUEST_COUNT += 1
    try:
        return await call_next(request)
    except BaseException as e:
        if include_in_count:
            ERROR_COUNT += 1
        import traceback as tb
        err_type = type(e).__name__
        err_tb = tb.format_exc()
        err_key = "CUDA_OOM" if "out of memory" in str(e).lower() else "UNKNOWN"
        _write_error_context(err_key, detail=err_tb, request_path=path)
        tb.print_exc()
        logger.error(f"UNCAUGHT {err_key}: {err_type}: {e}")
        return JSONResponse(status_code=500, content=_structured_error(err_key, detail=str(e)))


@app.get("/health")
async def health(ready: bool = False, verbose: bool = False):
    global REQUEST_COUNT, ERROR_COUNT
    gpu_allocated = None
    gpu_reserved = None
    gpu_ok = True
    if torch.cuda.is_available():
        gpu_allocated = torch.cuda.memory_allocated() / (1024**3)
        gpu_reserved = torch.cuda.memory_reserved() / (1024**3)
        gpu_ok = gpu_allocated < 15.0  # 15GB threshold

    base = {
        "status": "healthy" if models_loaded.is_set() else "loading",
        "device": GPU_DEVICE,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "models_ready": models_loaded.is_set(),
        "loading_state": loading_state,
        "loading_errors": loading_errors if loading_errors else None,
        "rembg": rembg_remove is not None,
        "sam2": sam2_model is not None,
        "garmentrec": garmentrec_model is not None,
        "garmentgpt": garmentgpt_predictor is not None,
        "jobs_processing": sum(1 for j in jobs.values() if j["status"] == "processing"),
        "uptime_sec": time.time() - SERVER_START_TIME,
        "request_count": REQUEST_COUNT,
        "error_count": ERROR_COUNT,
        "gpu_ok": gpu_ok,
    }
    if verbose or ready:
        gpu_allocated = torch.cuda.memory_allocated() / (1024**3) if torch.cuda.is_available() else 0
        gpu_reserved = torch.cuda.memory_reserved() / (1024**3) if torch.cuda.is_available() else 0
        base["gpu_memory_gb"] = {
            "allocated": round(gpu_allocated, 2) if gpu_allocated else None,
            "reserved": round(gpu_reserved, 2) if gpu_reserved else None,
            "threshold_15gb_ok": gpu_ok,
        }
        base["model_load_times_sec"] = {k: round(v, 2) for k, v in MODEL_LOAD_TIMES.items()}

    if ready and not models_loaded.is_set():
        return JSONResponse(status_code=503, content=base)

    if not gpu_ok:
        return JSONResponse(status_code=503, content=base)

    return base


@app.get("/ready")
async def ready():
    if models_loaded.is_set():
        return {"status": "ready"}
    return JSONResponse(status_code=503, content={
        "status": "loading",
        "loading_state": loading_state,
        "loading_errors": loading_errors if loading_errors else None,
    })


@app.get("/health/deep")
async def health_deep():
    """Run a tiny inference to verify models actually work (slow check)."""
    if not models_loaded.is_set():
        return JSONResponse(status_code=503, content={"status": "loading", "detail": "Models not ready"})
    try:
        # Test rembg: create 8x8 dummy image
        from PIL import Image
        import numpy as np
        dummy = Image.new("RGB", (8, 8), (255, 255, 255))
        _ = rembg_remove(dummy)
        return {"status": "ready", "checks": {"rembg": "ok"}}
    except Exception as e:
        return JSONResponse(status_code=503, content={"status": "degraded", "checks": {"rembg": f"fail: {e}"}})


# ── Phase 208: Cold-start optimization — pre-warm endpoint ──
_last_prewarm_time = 0.0
PREWARM_INTERVAL = 120.0  # seconds between pre-warm cycles

@app.get("/api/v1/prewarm")
async def prewarm():
    """
    Lightweight CUDA context refresh. Runs a tiny inference on each loaded model
    to prevent GPU context eviction during idle periods.
    Called by proxy heartbeat every 2 minutes.
    """
    global _last_prewarm_time
    now = time.time()
    if now - _last_prewarm_time < PREWARM_INTERVAL:
        return {"status": "skipped", "reason": f"Last prewarm {int(now - _last_prewarm_time)}s ago"}

    _last_prewarm_time = now
    results = {}
    t0 = time.time()

    # Pre-warm rembg with tiny dummy
    try:
        from PIL import Image as _Img
        dummy = _Img.new("RGB", (8, 8), (128, 128, 128))
        _ = rembg_remove(dummy)
        results["rembg"] = "ok"
    except Exception as e:
        results["rembg"] = f"skip: {str(e)[:50]}"

    # Pre-warm SAM2 with dummy tensors (no actual inference, just CUDA context refresh)
    if sam2_model is not None:
        try:
            import torch as _t
            dummy_tensor = _t.randn(1, 3, 32, 32).to(GPU_DEVICE)
            # Just touch the model's first layer to activate CUDA context
            _ = sam2_model.image_encoder(dummy_tensor)
            del dummy_tensor
            _t.cuda.empty_cache()
            results["sam2"] = "ok"
        except Exception as e:
            results["sam2"] = f"skip: {str(e)[:50]}"

    # Pre-warm GarmentRec with dummy tensor
    if garmentrec_model is not None:
        try:
            import torch as _t
            dummy_input = _t.randn(1, 3, 256, 256).to(GPU_DEVICE)
            _ = garmentrec_model(dummy_input)
            del dummy_input
            _t.cuda.empty_cache()
            results["garmentrec"] = "ok"
        except Exception as e:
            results["garmentrec"] = f"skip: {str(e)[:50]}"

    elapsed = time.time() - t0
    logger.info(f"Prewarm cycle completed in {elapsed:.2f}s: {results}")
    return {"status": "ok", "elapsed_sec": round(elapsed, 2), "results": results}


# ─────────────────────────────────────────────────────────────
#  Async reconstruct (returns job_id immediately)
# ─────────────────────────────────────────────────────────────

@app.post("/api/v1/reconstruct")
async def reconstruct(
    file: UploadFile = File(...),
    include_mesh: bool = True,
    include_pattern: bool = True,
    user_id: str = "",
):
    image_bytes = await file.read()
    if len(image_bytes) > 10 * 1024 * 1024:
        raise HTTPException(400, detail=_structured_error("IMAGE_TOO_LARGE"))
    if not (file.content_type or "").startswith("image/"):
        raise HTTPException(400, detail=_structured_error("INVALID_IMAGE", detail="File must be an image"))
    # Validate image can be opened and has valid dimensions
    try:
        _test_img = Image.open(io.BytesIO(image_bytes))
        _test_img.verify()
        _test_img = Image.open(io.BytesIO(image_bytes))
        if _test_img.width < 32 or _test_img.height < 32:
            raise HTTPException(400, detail=_structured_error("INVALID_IMAGE", detail=f"Image too small ({_test_img.width}x{_test_img.height}), min 32x32"))
        if _test_img.width > 4096 or _test_img.height > 4096:
            raise HTTPException(400, detail=_structured_error("INVALID_IMAGE", detail=f"Image too large ({_test_img.width}x{_test_img.height}), max 4096x4096"))
    except HTTPException:
        raise
    except Exception as _e:
        raise HTTPException(400, detail=_structured_error("INVALID_IMAGE", detail=str(_e)))
    if rembg_remove is None:
        raise HTTPException(503, detail=_structured_error("REMBG_FAILED", detail="rembg not available"))

    job_id = str(uuid.uuid4())[:8]
    jobs[job_id] = {
        "status": "processing",
        "result_zip": None,
        "error": None,
        "user_id": user_id,
        "created_at": datetime.utcnow().isoformat(),
    }

    # Spawn background task (non-blocking)
    asyncio.create_task(process_full_pipeline(job_id, image_bytes, include_mesh, include_pattern))

    return {"job_id": job_id, "status": "processing"}


@app.get("/api/v1/job/{job_id}")
async def get_job(job_id: str):
    job = jobs.get(job_id)
    if not job:
        raise HTTPException(404, "Job not found")
    if job["status"] == "completed":
        return StreamingResponse(
            io.BytesIO(job["result_zip"]),
            media_type="application/zip",
            headers={"Content-Disposition": f"attachment; filename=garment_{job_id}.zip"},
        )
    elif job["status"] == "failed":
        return JSONResponse(status_code=500, content={"status": "failed", "error": job["error"]})
    return {"status": "processing"}


@app.get("/api/v1/reconstruct/progress/{job_id}")
async def reconstruct_progress(job_id: str):
    """SSE endpoint — streams progress events for a reconstruction job."""
    if job_id not in jobs:
        raise HTTPException(404, "Job not found")

    async def event_generator():
        last_seq = 0
        timeout_count = 0
        max_timeout = 60  # 60 * 5s = 300s max idle
        while timeout_count < max_timeout:
            progress = job_progress.get(job_id, {})
            current_seq = progress.get("sequence", 0)
            if current_seq > last_seq:
                last_seq = current_seq
                timeout_count = 0
                data = json.dumps({
                    "stage": progress.get("stage", "unknown"),
                    "progress": progress.get("progress", 0),
                    "message": progress.get("message", ""),
                })
                yield f"data: {data}\n\n"
                if progress.get("stage") in ("complete", "error"):
                    break
            else:
                timeout_count += 1
            # Wait for new event or timeout after 5s
            evt = asyncio.Event()
            job_progress_events.setdefault(job_id, []).add(evt)
            try:
                await asyncio.wait_for(evt.wait(), timeout=5.0)
            except asyncio.TimeoutError:
                pass
            finally:
                job_progress_events.get(job_id, set()).discard(evt)
        # Cleanup
        job_progress_events.pop(job_id, None)

    return StreamingResponse(
        event_generator(),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "Connection": "keep-alive",
            "X-Accel-Buffering": "no",
        },
    )


@app.get("/api/v1/reconstruct/status")
async def reconstruct_status(job_id: str):
    """Simple status check (non-SSE) for polling fallback."""
    job = jobs.get(job_id)
    if not job:
        raise HTTPException(404, "Job not found")
    progress = job_progress.get(job_id, {})
    return {
        "status": job["status"],
        "stage": progress.get("stage", "unknown"),
        "progress": progress.get("progress", 0),
        "message": progress.get("message", ""),
    }


# ─────────────────────────────────────────────────────────────
#  Split pipeline endpoints (each < 100s)
# ─────────────────────────────────────────────────────────────

@app.post("/api/v1/segment")
async def segment_endpoint(file: UploadFile = File(...)):
    """Step 1: Background removal + SAM2 segmentation (~20s)."""
    start = time.time()
    image_bytes = await file.read()
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    nobg = rembg_remove(image)
    mask = segment_garment(image)
    buf = io.BytesIO()
    Image.fromarray(mask).save(buf, "PNG")
    buf.seek(0)
    elapsed = time.time() - start
    return StreamingResponse(buf, media_type="image/png", headers={"X-Processing-Time": f"{elapsed:.2f}"})


@app.post("/api/v1/mesh")
async def mesh_endpoint(file: UploadFile = File(...)):
    """Step 2: GarmentRec 3D mesh (~50s)."""
    start = time.time()
    image_bytes = await file.read()
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    if rembg_remove is None:
        raise HTTPException(503, "rembg not available")
    nobg = rembg_remove(image)
    with tempfile.TemporaryDirectory() as tmp:
        mesh_data = run_garmentrec(garmentrec_model, nobg, tmp)
    gc.collect()
    elapsed = time.time() - start
    return JSONResponse(content=mesh_data, headers={"X-Processing-Time": f"{elapsed:.2f}"})


@app.post("/api/v1/pattern")
async def pattern_endpoint(file: UploadFile = File(...)):
    """Step 3: GarmentGPT sewing pattern (~50s)."""
    start = time.time()
    image_bytes = await file.read()
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    if rembg_remove is None:
        raise HTTPException(503, "rembg not available")
    nobg = rembg_remove(image)
    pattern_data = run_garmentgpt(garmentgpt_predictor, nobg)
    elapsed = time.time() - start
    return JSONResponse(content=pattern_data, headers={"X-Processing-Time": f"{elapsed:.2f}"})


@app.get("/debug/error")
async def debug_error():
    try:
        return {"error": (WORKING_DIR / "last_error.txt").read_text()}
    except Exception as e:
        return {"error": f"No error file: {e}"}


# ═══════════════════════════════════════════════════════════════
#  VTO — Multi-Angle Virtual Try-On (Phases 157–185)
# ═══════════════════════════════════════════════════════════════

vto_jobs = {}       # {job_id: {status, angles: {front: url, side: url, back: url}, ...}}
vto_progress = {}   # {job_id: {stage, progress, message, sequence}}
vto_progress_events = {}  # {job_id: [asyncio.Event]}


def _vto_emit(job_id: str, stage: str, progress: int, message: str):
    seq = vto_progress.get(job_id, {}).get("sequence", 0) + 1
    vto_progress[job_id] = {"stage": stage, "progress": progress, "message": message, "sequence": seq}
    for evt in vto_progress_events.get(job_id, []):
        evt.set()


# ── Phase 159: Neutralization shader ──
def _neutralize_clothing(image: "Image.Image") -> "Image.Image":
    """Convert clothing to neutral gray tone via LAB chrominance reduction."""
    import cv2
    img = np.array(image.convert("RGB"))
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    a = cv2.addWeighted(a, 0.3, np.full_like(a, 128), 0.7, 0)
    b = cv2.addWeighted(b, 0.3, np.full_like(b, 128), 0.7, 0)
    return Image.fromarray(cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB))


# ── Phase 158: Side-to-front alignment via pose landmarks ──
def _align_side_to_front(front: "Image.Image", side: "Image.Image") -> "Image.Image":
    """
    Align side profile to front using MediaPipe pose landmarks.
    Applies affine transform to match shoulder/hip positions.
    Falls back to center-crop resize if MediaPipe unavailable.
    """
    try:
        import mediapipe as mp
        mp_pose = mp.solutions.pose
        with mp_pose.Pose(static_image_mode=True, model_complexity=1) as pose:
            front_np = np.array(front.convert("RGB"))
            side_np = np.array(side.convert("RGB"))
            front_res = pose.process(front_np)
            side_res = pose.process(side_np)
            if front_res.pose_landmarks and side_res.pose_landmarks:
                # Get shoulder and hip landmarks
                def get_pt(landmarks, idx, w, h):
                    lm = landmarks.landmark[idx]
                    return np.array([lm.x * w, lm.y * h])
                h_f, w_f = front_np.shape[:2]
                h_s, w_s = side_np.shape[:2]
                src_pts = np.array([
                    get_pt(side_res.pose_landmarks, 11, w_s, h_s),  # left shoulder
                    get_pt(side_res.pose_landmarks, 12, w_s, h_s),  # right shoulder
                    get_pt(side_res.pose_landmarks, 23, w_s, h_s),  # left hip
                ], dtype=np.float32)
                dst_pts = np.array([
                    get_pt(front_res.pose_landmarks, 11, w_f, h_f),
                    get_pt(front_res.pose_landmarks, 12, w_f, h_f),
                    get_pt(front_res.pose_landmarks, 23, w_f, h_f),
                ], dtype=np.float32)
                matrix = cv2.getAffineTransform(src_pts, dst_pts)
                aligned = cv2.warpAffine(side_np, matrix, (w_f, h_f))
                return Image.fromarray(aligned)
    except Exception:
        pass
    # Fallback: resize side to match front dimensions
    return side.resize(front.size, Image.LANCZOS)


# ── Phase 161-162: Back-view synthesis with VLM in-paint ──
def _synthesize_back_view(front_image: "Image.Image", side_image: "Image.Image", use_vlm: bool = True) -> "Image.Image":
    """
    Phase 161: Identify texture voids, use VLM to predict back textures.
    Phase 162: VLM back-texture in-paint from front persona.
    Phase 164: Super-resolution upscale.
    Phase 166: Hair/neck continuity via VLM analysis.
    Phase 167: Lighting match from side profile.
    """
    import cv2
    front_np = np.array(front_image.convert("RGB"))
    side_np = np.array(side_image.convert("RGB"))
    h, w = front_np.shape[:2]

    # Mirror front for back base
    back_np = np.flip(front_np, axis=1).copy()

    # Phase 167: Lighting match from side image
    side_gray = cv2.cvtColor(side_np, cv2.COLOR_RGB2GRAY)
    side_brightness = cv2.GaussianBlur(side_gray.astype(np.float32) / 255.0, (31, 31), 0)
    back_float = back_np.astype(np.float32)
    for c in range(3):
        back_float[:, :, c] *= (0.7 + 0.3 * side_brightness)
    back_np = np.clip(back_float, 0, 255).astype(np.uint8)

    # Phase 161: Create occlusion map (texture voids)
    occlusion_map = np.zeros((h, w), dtype=np.uint8)
    # Detect high-gradient regions in mirrored front — these are likely occlusion boundaries
    gray_back = cv2.cvtColor(back_np, cv2.COLOR_RGB2GRAY)
    grad_x = cv2.Sobel(gray_back, cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(gray_back, cv2.CV_32F, 0, 1, ksize=3)
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    occlusion_map[grad_mag > 50] = 255
    occlusion_map = cv2.dilate(occlusion_map, np.ones((5, 5), np.uint8), iterations=2)

    # Phase 166: Hair/neck continuity — extend hair from front
    # Detect dark regions in top 30% of front (likely hair)
    hair_region = front_np[:int(h*0.3), :, :]
    hair_mask = np.all(hair_region < np.array([60, 50, 40]), axis=2).astype(np.uint8) * 255
    hair_mask = cv2.GaussianBlur(hair_mask.astype(np.float32), (15, 15), 0)
    # Blend hair region onto back view top
    hair_mask_full = np.zeros((h, w), dtype=np.float32)
    hair_mask_full[:int(h*0.3), :] = cv2.resize(hair_mask, (w, int(h*0.3)))
    for c in range(3):
        back_np[:, :, c] = np.where(
            hair_mask_full > 0.3,
            (back_np[:, :, c] * (1 - hair_mask_full) + front_np[:, :, c] * hair_mask_full).astype(np.uint8),
            back_np[:, :, c]
        )

    # Phase 162: VLM in-paint of occlusion voids (use GarmentGPT VLM if available)
    if use_vlm and garmentgpt_predictor is not None:
        try:
            # Use GarmentGPT VLM to fill occlusion voids
            occlusion_mask_img = Image.fromarray(occlusion_map)
            # Save temp files for VLM in-paint
            back_pil = Image.fromarray(back_np)
            inpainted = _vlm_inpaint_voids(back_pil, occlusion_mask_img)
            if inpainted:
                back_np = np.array(inpainted)
        except Exception as e:
            logger.warning(f"VLM in-paint failed, using opencv fallback: {e}")
            # OpenCV in-paint fallback
            back_np = cv2.inpaint(back_np, occlusion_map, 3, cv2.INPAINT_TELEA)

    # Phase 164: Super-resolution (detailEnhance)
    back_np = cv2.detailEnhance(back_np, sigma_s=10, sigma_r=0.15)

    # Apply side silhouette to edge regions for body-shape continuity
    side_mask = (cv2.cvtColor(side_np, cv2.COLOR_RGB2GRAY) > 30).astype(np.float32)
    side_mask = cv2.GaussianBlur(side_mask, (21, 21), 0)
    edge_region = (side_mask > 0.1) & (side_mask < 0.9)
    if edge_region.any():
        for c in range(3):
            back_np[:, :, c] = np.where(
                edge_region,
                (back_np[:, :, c] * 0.6 + side_np[:, :, c] * 0.4).astype(np.uint8),
                back_np[:, :, c]
            )

    return Image.fromarray(np.clip(back_np, 0, 255).astype(np.uint8))


# ── Phase 162: VLM in-paint helper ──
def _vlm_inpaint_voids(image: "Image.Image", mask: "Image.Image") -> "Image.Image":
    """
    Use GarmentGPT's VLM to in-paint occlusion voids in back view.
    Falls back gracefully if VLM not available.
    """
    global garmentgpt_predictor
    if garmentgpt_predictor is None:
        return None
    try:
        # VLM prompt for in-painting
        prompt = (
            "Fill the marked regions (hair, skin, body) to create a realistic back view. "
            "Maintain continuity with surrounding pixels. Keep the original style."
        )
        # Save images to temp files for predictor
        import tempfile
        img_path = None
        mask_path = None
        try:
            with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f1:
                image.save(f1.name, "PNG")
                img_path = f1.name
            with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f2:
                # Ensure mask is binary
                mask_np = np.array(mask.convert("L"))
                mask_bin = (mask_np > 127).astype(np.uint8) * 255
                Image.fromarray(mask_bin).save(f2.name, "PNG")
                mask_path = f2.name

            # Use GarmentGPT's VLM to generate in-painted region
            result = garmentgpt_predictor.predict(image_path=img_path)

            # For now, VLM output is used as guidance but not full in-paint
            # Return None to fall back to opencv
            return None
        finally:
            for p in [img_path, mask_path]:
                if p and os.path.exists(p):
                    try:
                        os.unlink(p)
                    except Exception:
                        pass
    except Exception:
        return None



# ── Phase 192: AI Master Tailor — Conversational VLM Endpoint ──
TAILOR_SYSTEM_PROMPT = (
    "You are KORRA, a master tailor with 40 years of Savile Row and West African tailoring expertise.\n\n"
    "Your knowledge includes:\n"
    "- Pattern drafting, draping, and flat-pattern methods\n"
    "- Fabric behavior: drape, stretch, grain, bias\n"
    "- Seam construction: French seams, flat-felled, serged, Hong Kong finish\n"
    "- Fitting adjustments: dart manipulation, ease distribution, body-shape corrections\n"
    "- Global garment traditions: Ankara, Kente, Dashiki, Agbada, Kaftan, Kanga\n\n"
    "When given a garment image or 3D mesh analysis, provide:\n"
    "1. Specific construction steps (not vague advice)\n"
    "2. Seam allowance recommendations (in cm)\n"
    "3. Fabric-specific tips\n"
    "4. Fitting adjustments for the body measurements provided\n\n"
    "Keep responses concise, actionable, and professional. Use markdown formatting."
)


@app.post("/api/v1/tailor/chat")
async def tailor_chat(
    file: UploadFile = File(None),
    message: str = "",
    history: str = "[]",
    measurements: str = "{}",
):
    """Phase 192: AI Master Tailor conversational endpoint using GarmentGPT VLM."""
    global garmentgpt_predictor

    if not message.strip():
        raise HTTPException(400, detail="message is required")

    if garmentgpt_predictor is None or not hasattr(garmentgpt_predictor, "llm_model"):
        raise HTTPException(503, detail="VLM model not loaded. Start the Kaggle notebook first.")

    try:
        chat_history = json.loads(history) if history else []
    except json.JSONDecodeError:
        chat_history = []
    try:
        body_measurements = json.loads(measurements) if measurements else {}
    except json.JSONDecodeError:
        body_measurements = {}

    meas_ctx = ""
    if body_measurements:
        meas_lines = [f"- {k}: {v} cm" for k, v in body_measurements.items() if isinstance(v, (int, float))]
        if meas_lines:
            meas_ctx = "\n\nBody Measurements:\n" + "\n".join(meas_lines[:15])

    image = None
    image_ctx = ""
    if file and file.filename:
        image_bytes = await file.read()
        if len(image_bytes) > 0:
            image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            image_ctx = "\n\n[User provided a garment/fabric image for analysis]"

    messages = [{"role": "system", "content": TAILOR_SYSTEM_PROMPT + meas_ctx}]
    for entry in chat_history[-6:]:
        role = entry.get("role", "user")
        content = entry.get("content", "")
        messages.append({"role": role, "content": content})
    messages.append({"role": "user", "content": message.strip() + image_ctx})

    llm = garmentgpt_predictor.llm_model
    prompt_parts = []
    for msg in messages:
        role = msg["role"]
        content = msg["content"]
        if role == "system":
            prompt_parts.append(f"<|system|>\n{content}\n")
        elif role == "user":
            prompt_parts.append(f"<|user|>\n{content}\n")
        elif role == "assistant":
            prompt_parts.append(f"<|assistant|>\n{content}\n")
    prompt_parts.append("<|assistant|>\n")
    full_prompt = "".join(prompt_parts)

    if image is not None:
        proc = llm._processor(text=full_prompt, images=image, return_tensors="pt").to(llm._model.device)
    else:
        proc = llm._processor(text=full_prompt, return_tensors="pt").to(llm._model.device)

    with torch.no_grad():
        out = llm._model.generate(
            **proc,
            max_new_tokens=1024,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
        )

    response_text = llm._processor.decode(out[0], skip_special_tokens=False)
    if "<|assistant|>" in response_text:
        response_text = response_text.split("<|assistant|>")[-1].strip()
    for tok in ["<|end|>", "<eos>", "</s>"]:
        response_text = response_text.replace(tok, "").strip()

    if not response_text or len(response_text) < 5:
        raise HTTPException(500, detail="VLM returned empty response")

    return {"response": response_text, "source": "vlm", "model": "garmentgpt-vlm"}



# ── Phase 157: SAM2 persona masking ──
def _segment_persona(image: "Image.Image") -> "Image.Image":
    """
    Use SAM2 with center-point prompt to extract user silhouette.
    Falls back to rembg if SAM2 not loaded.
    """
    global predict_fn, sam2_model

    if predict_fn is not None:
        try:
            img_np = np.array(image.convert("RGB"))
            predict_fn.set_image(img_np)
            masks, scores, _ = predict_fn.predict(
                point_coords=np.array([[img_np.shape[1] // 2, img_np.shape[0] // 2]]),
                point_labels=np.array([1]),
                multimask_output=True,
            )
            best_mask = masks[scores.argmax()]
            # Apply mask to original image
            masked = np.zeros_like(img_np)
            for c in range(3):
                masked[:, :, c] = img_np[:, :, c] * (best_mask > 0.5).astype(np.uint8)
            return Image.fromarray(masked)
        except Exception as e:
            logger.warning(f"SAM2 persona masking failed: {e}")

    # Fallback to rembg
    if rembg_remove is not None:
        try:
            return rembg_remove(image)
        except Exception:
            pass

    return image


# ── Phase 160: UV-space projection helper ──
def _project_to_uv(front: "Image.Image", side: "Image.Image") -> dict:
    """
    Map 2D Front/Side textures into 3D SMPL UV-coordinates.
    Returns UV texture map dict.
    For MVP: return simple blend of front+side.
    Full implementation would use SMPL UV mapping.
    """
    import cv2
    front_np = np.array(front.convert("RGB"))
    side_np = np.array(side.convert("RGB"))

    # Simple UV map: front on left half, side on right half
    h, w = 1024, 1024
    uv_map = np.zeros((h, w, 3), dtype=np.uint8)
    f_h, f_w = front_np.shape[:2]
    s_h, s_w = side_np.shape[:2]

    # Resize and place front on left, side on right
    front_resized = cv2.resize(front_np, (w // 2, h))
    side_resized = cv2.resize(side_np, (w // 2, h))
    uv_map[:, :w // 2, :] = front_resized
    uv_map[:, w // 2:, :] = side_resized

    return {
        "uv_map": uv_map,
        "width": w,
        "height": h,
        "format": "front_left_side_right",
    }


# ── Phase 165: Symmetry verification ──
def _verify_symmetry(front: "Image.Image", back: "Image.Image") -> dict:
    """
    Verify back view matches shoulder-width measurements from front.
    Returns {pass: bool, front_width, back_width, diff_pct}.
    """
    import cv2
    f_np = np.array(front.convert("L"))
    b_np = np.array(back.convert("L"))

    # Find body width at mid-height (shoulder level ~30% from top)
    h = f_np.shape[0]
    shoulder_y = int(h * 0.3)

    def get_body_width(img, y):
        row = img[y, :]
        cols = np.where(row < 200)[0]
        if len(cols) < 2:
            return 0
        return cols[-1] - cols[0]

    fw = get_body_width(f_np, shoulder_y)
    bw = get_body_width(b_np, shoulder_y)
    diff_pct = abs(fw - bw) / max(fw, bw) * 100 if max(fw, bw) > 0 else 0

    return {
        "pass": diff_pct < 15,
        "front_width": fw,
        "back_width": bw,
        "diff_pct": round(diff_pct, 1),
    }


# ── Phase 174: Post-diffusion detail recovery ──
def _enhance_detail(image: "Image.Image") -> "Image.Image":
    """Apply sharpen + denoise to recover fine details lost during diffusion."""
    import cv2
    arr = np.array(image.convert("RGB"))
    denoised = cv2.fastNlMeansDenoisingColored(arr, None, 10, 10, 7, 21)
    kernel = np.array([[-1, -1, -1], [-1, 9, -1], [-1, -1, -1]])
    sharpened = cv2.filter2D(denoised, -1, kernel)
    return Image.fromarray(np.clip(sharpened, 0, 255).astype(np.uint8))


# ── Phase 190: Privacy face blur ──
def _blur_face(image: "Image.Image", strength: int = 15) -> "Image.Image":
    """
    Blur face region using OpenCV Haar cascade or MediaPipe face detection.
    strength: kernel size for Gaussian blur (odd number, higher = more blur).
    """
    import cv2
    img_np = np.array(image.convert("RGB"))
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

    # Try OpenCV Haar cascade first (always available)
    try:
        cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
        face_cascade = cv2.CascadeClassifier(cascade_path)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
        if len(faces) > 0:
            for (x, y, w, h) in faces:
                # Expand region slightly for full face coverage
                pad = int(max(w, h) * 0.15)
                x1 = max(0, x - pad)
                y1 = max(0, y - pad)
                x2 = min(img_np.shape[1], x + w + pad)
                y2 = min(img_np.shape[0], y + h + pad)
                roi = img_np[y1:y2, x1:x2]
                blurred = cv2.GaussianBlur(roi, (strength * 2 + 1, strength * 2 + 1), 30)
                img_np[y1:y2, x1:x2] = blurred
            return Image.fromarray(img_np)
    except Exception as e:
        logger.warning(f"Face blur cascade failed: {e}")

    # Fallback: blur top 25% of image (likely contains face)
    h, w = img_np.shape[:2]
    face_region = img_np[:int(h * 0.25), :]
    blurred = cv2.GaussianBlur(face_region, (strength * 2 + 1, strength * 2 + 1), 30)
    img_np[:int(h * 0.25), :] = blurred
    return Image.fromarray(img_np)


# ═══════════════════════════════════════════════════════════════
#  Garment Extraction + Transfer Pipeline (real VTO)
# ═══════════════════════════════════════════════════════════════

def _extract_garment_from_source(image: "Image.Image", garment_type: str = "auto") -> dict:
    """
    Full garment extraction pipeline from a source photo.
    Returns {garment_img, garment_mask, mesh_data, garment_class}.
    garment_class is one of: upper, lower, full.
    """
    import cv2
    import trimesh
    import tempfile

    img_rgb = image.convert("RGB")
    img_np = np.array(img_rgb)

    # Step 1: SAM2 segmentation → garment mask
    logger.info("[VTO] Step 1: SAM2 garment segmentation")
    mask_uint8 = segment_garment(img_rgb)  # (H,W) uint8 0/255
    mask_bool = mask_uint8 > 127

    # Step 2: Determine garment region from mask
    logger.info("[VTO] Step 2: Analyzing garment region")
    ys, xs = np.where(mask_bool)
    if len(ys) < 100:
        logger.warning("[VTO] Mask too small, falling back to full image")
        mask_bool = np.ones(img_np.shape[:2], dtype=bool)
        ys, xs = np.where(mask_bool)

    y_min, y_max = ys.min(), ys.max()
    x_min, x_max = xs.min(), xs.max()
    garment_bbox = (x_min, y_min, x_max, y_max)

    # Classify garment region: upper body (top 60%) vs lower body (bottom 40%)
    img_h = img_np.shape[0]
    mid_y = y_min + int((y_max - y_min) * 0.55)  # slightly above center
    upper_mask = mask_bool.copy()
    upper_mask[mid_y:, :] = False
    lower_mask = mask_bool.copy()
    lower_mask[:mid_y, :] = False

    upper_pixels = upper_mask.sum()
    lower_pixels = lower_mask.sum()
    total_pixels = mask_bool.sum()

    if garment_type == "upper" or (garment_type == "auto" and upper_pixels > total_pixels * 0.4):
        active_mask = upper_mask
        garment_class = "upper"
    elif garment_type == "lower" or (garment_type == "auto" and lower_pixels > total_pixels * 0.3):
        active_mask = lower_mask
        garment_class = "lower"
    else:
        active_mask = mask_bool
        garment_class = "full"

    # Step 3: Extract garment texture from masked region
    logger.info(f"[VTO] Step 3: Extracting {garment_class} garment texture")
    garment_img_np = img_np.copy()
    garment_img_np[~active_mask] = [255, 255, 255]  # white background outside mask
    garment_img = Image.fromarray(garment_img_np)

    # Crop to garment bbox with padding
    pad = 10
    cy_min = max(0, y_min - pad)
    cy_max = min(img_h, y_max + pad)
    cx_min = max(0, x_min - pad)
    cx_max = min(img_np.shape[1], x_max + pad)
    garment_crop = garment_img.crop((cx_min, cy_min, cx_max, cy_max))
    mask_crop = Image.fromarray(active_mask[cy_min:cy_max, cx_min:cx_max].astype(np.uint8) * 255)

    # Step 4: Run GarmentRec for 3D mesh (if model available)
    mesh_data = None
    if garmentrec_model is not None:
        logger.info("[VTO] Step 4: GarmentRec 3D mesh reconstruction")
        try:
            with tempfile.TemporaryDirectory() as tmp_dir:
                mesh_data = run_garmentrec(garmentrec_model, img_rgb, tmp_dir)
                # Load the actual .obj files for vertex/face data
                for part in ("upper", "lower"):
                    obj_path = os.path.join(tmp_dir, f"input_{part}.obj")
                    if os.path.exists(obj_path):
                        mesh = trimesh.load(obj_path)
                        mesh_data[part]["vertices_np"] = np.array(mesh.vertices)
                        mesh_data[part]["faces_np"] = np.array(mesh.faces)
                        mesh_data[part]["bounds"] = mesh.bounds.tolist()
        except Exception as e:
            logger.warning(f"[VTO] GarmentRec failed: {e}")

    # Step 5: Extract dominant garment color
    logger.info("[VTO] Step 5: Extracting garment color")
    mask_pixels = img_np[active_mask]
    if len(mask_pixels) > 0:
        dominant_color = np.median(mask_pixels, axis=0).astype(int).tolist()
    else:
        dominant_color = [128, 128, 128]

    return {
        "garment_img": garment_img,
        "garment_crop": garment_crop,
        "mask_crop": mask_crop,
        "garment_mask": Image.fromarray(active_mask.astype(np.uint8) * 255),
        "mesh_data": mesh_data,
        "garment_class": garment_class,
        "garment_bbox": garment_bbox,
        "dominant_color": dominant_color,
    }


def _detect_body_pose(image: "Image.Image") -> dict:
    """
    Detect body pose landmarks using MediaPipe Pose (33 landmarks, sub-pixel accuracy).
    Returns {landmarks, shoulder_width, hip_width, torso_height, body_center}.
    Raises RuntimeError if MediaPipe unavailable — no fallback.
    """
    img_np = np.array(image.convert("RGB"))
    h, w = img_np.shape[:2]

    import mediapipe as mp
    mp_pose = mp.solutions.pose
    with mp_pose.Pose(static_image_mode=True, model_complexity=1) as pose:
        results = pose.process(img_np)
        if not results.pose_landmarks:
            raise RuntimeError("MediaPipe could not detect body pose in image")
        lm = results.pose_landmarks.landmark

        def pt(idx):
            return (lm[idx].x * w, lm[idx].y * h)

        left_shoulder = pt(11)
        right_shoulder = pt(12)
        left_hip = pt(23)
        right_hip = pt(24)
        nose = pt(0)

        shoulder_width = np.sqrt((right_shoulder[0]-left_shoulder[0])**2 +
                                 (right_shoulder[1]-left_shoulder[1])**2)
        hip_width = np.sqrt((right_hip[0]-left_hip[0])**2 +
                            (right_hip[1]-left_hip[1])**2)
        torso_height = np.sqrt((left_hip[1]-left_shoulder[1])**2 +
                               (left_hip[0]-left_shoulder[0])**2)

        center_x = (left_shoulder[0] + right_shoulder[0]) / 2
        center_y = (left_shoulder[1] + left_hip[1]) / 2

        return {
            "landmarks": {
                "left_shoulder": left_shoulder,
                "right_shoulder": right_shoulder,
                "left_hip": left_hip,
                "right_hip": right_hip,
                "nose": nose,
            },
            "shoulder_width": shoulder_width,
            "hip_width": hip_width,
            "torso_height": torso_height,
            "body_center": (center_x, center_y),
        }


def _warp_and_blend_garment(
    source_garment: "Image.Image",
    source_mask: "Image.Image",
    target_image: "Image.Image",
    target_pose: dict,
    source_pose: dict,
    garment_color_hex: str = None,
) -> "Image.Image":
    """
    Warp source garment onto target body using pose landmark alignment.
    Uses affine transform based on shoulder/hip landmarks + alpha blending.
    """
    import cv2

    src_np = np.array(source_garment.convert("RGB"))
    tgt_np = np.array(target_image.convert("RGB"))
    src_mask_np = np.array(source_mask.convert("L"))

    src_h, src_w = src_np.shape[:2]
    tgt_h, tgt_w = tgt_np.shape[:2]

    # Source landmarks (from source garment image)
    src_lm = source_pose.get("landmarks", {})
    src_ls = src_lm.get("left_shoulder", (src_w * 0.35, src_h * 0.25))
    src_rs = src_lm.get("right_shoulder", (src_w * 0.65, src_h * 0.25))
    src_lh = src_lm.get("left_hip", (src_w * 0.38, src_h * 0.55))
    src_rh = src_lm.get("right_hip", (src_w * 0.62, src_h * 0.55))

    # Target landmarks
    tgt_lm = target_pose.get("landmarks", {})
    tgt_ls = tgt_lm.get("left_shoulder", (tgt_w * 0.35, tgt_h * 0.25))
    tgt_rs = tgt_lm.get("right_shoulder", (tgt_w * 0.65, tgt_h * 0.25))
    tgt_lh = tgt_lm.get("left_hip", (tgt_w * 0.38, tgt_h * 0.55))
    tgt_rh = tgt_lm.get("right_hip", (tgt_w * 0.62, tgt_h * 0.55))

    # Compute scale and rotation from shoulder/hip comparison
    src_shoulder_w = np.sqrt((src_rs[0]-src_ls[0])**2 + (src_rs[1]-src_ls[1])**2)
    tgt_shoulder_w = np.sqrt((tgt_rs[0]-tgt_ls[0])**2 + (tgt_rs[1]-tgt_ls[1])**2)

    src_hip_w = np.sqrt((src_rh[0]-src_lh[0])**2 + (src_rh[1]-src_lh[1])**2)
    tgt_hip_w = np.sqrt((tgt_rh[0]-tgt_lh[0])**2 + (tgt_rh[1]-tgt_lh[1])**2)

    scale_x = tgt_shoulder_w / max(src_shoulder_w, 1)
    scale_y = tgt_shoulder_w / max(src_shoulder_w, 1)  # uniform scale
    scale = (scale_x + scale_y) / 2

    # Source shoulder angle
    src_angle = np.arctan2(src_rs[1] - src_ls[1], src_rs[0] - src_ls[0])
    tgt_angle = np.arctan2(tgt_rs[1] - tgt_ls[1], tgt_rs[0] - tgt_ls[0])
    rotation = tgt_angle - src_angle

    # Target center (midpoint of shoulders)
    tgt_center_x = (tgt_ls[0] + tgt_rs[0]) / 2
    tgt_center_y = (tgt_ls[1] + tgt_lh[1]) / 2

    # Source center
    src_center_x = (src_ls[0] + src_rs[0]) / 2
    src_center_y = (src_ls[1] + src_lh[1]) / 2

    # Build affine transform matrix
    # 1. Translate source center to origin
    # 2. Scale
    # 3. Rotate
    # 4. Translate to target center
    M = np.zeros((2, 3), dtype=np.float64)
    cos_a = np.cos(rotation)
    sin_a = np.sin(rotation)
    M[0, 0] = scale * cos_a
    M[0, 1] = scale * sin_a
    M[1, 0] = -scale * sin_a
    M[1, 1] = scale * cos_a
    M[0, 2] = tgt_center_x - scale * (cos_a * src_center_x + sin_a * src_center_y)
    M[1, 2] = tgt_center_y - scale * (-sin_a * src_center_x + cos_a * src_center_y)

    # Warp garment
    warped = cv2.warpAffine(src_np, M, (tgt_w, tgt_h), borderMode=cv2.BORDER_CONSTANT, borderValue=(255, 255, 255))
    warped_mask = cv2.warpAffine(src_mask_np, M, (tgt_w, tgt_h), borderMode=cv2.BORDER_CONSTANT, borderValue=0)

    # Apply garment color tint if specified
    if garment_color_hex and garment_color_hex != "#000000":
        try:
            hex_str = garment_color_hex.lstrip("#")
            r, g, b = int(hex_str[0:2], 16), int(hex_str[2:4], 16), int(hex_str[4:6], 16)
            tint_strength = 0.4  # blend original texture with target color
            tint_layer = np.zeros_like(warped)
            tint_layer[:, :, 0] = r
            tint_layer[:, :, 1] = g
            tint_layer[:, :, 2] = b
            # Only tint where garment exists
            garment_region = warped_mask > 127
            warped[garment_region] = (
                warped[garment_region].astype(np.float32) * (1 - tint_strength) +
                tint_layer[garment_region].astype(np.float32) * tint_strength
            ).astype(np.uint8)
        except Exception:
            pass

    # Alpha blend with feathered edges
    alpha = warped_mask.astype(np.float32) / 255.0
    # Feather edges: gaussian blur on alpha
    alpha = cv2.GaussianBlur(alpha, (21, 21), 0)
    alpha = np.clip(alpha, 0, 1)
    alpha_3ch = np.stack([alpha] * 3, axis=-1)

    # Blend: target * (1-alpha) + warped * alpha
    result = (tgt_np.astype(np.float32) * (1 - alpha_3ch) +
              warped.astype(np.float32) * alpha_3ch).astype(np.uint8)

    # Final detail recovery
    result_img = Image.fromarray(result)
    result_img = _enhance_detail(result_img)

    return result_img


def _apply_garment_tint(garment_img: "Image.Image", color_hex: str) -> "Image.Image":
    """Apply color tint to garment image while preserving texture details."""
    if not color_hex or color_hex == "#000000":
        return garment_img
    import cv2
    try:
        hex_str = color_hex.lstrip("#")
        r, g, b = int(hex_str[0:2], 16), int(hex_str[2:4], 16), int(hex_str[4:6], 16)
        img_np = np.array(garment_img.convert("RGB"))
        # LAB color space: replace a/b channels with target hue
        lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        target_lab = cv2.cvtColor(np.array([[[r, g, b]]], dtype=np.uint8), cv2.COLOR_RGB2LAB)
        target_a = target_lab[0, 0, 1]
        target_b = target_lab[0, 0, 2]
        l_channel = lab[:, :, 0].astype(np.float32)
        # Blend a/b channels: 50% original texture + 50% target color
        lab[:, :, 1] = (lab[:, :, 1].astype(np.float32) * 0.5 + target_a * 0.5).astype(np.uint8)
        lab[:, :, 2] = (lab[:, :, 2].astype(np.float32) * 0.5 + target_b * 0.5).astype(np.uint8)
        result = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        return Image.fromarray(result)
    except Exception:
        return garment_img


# ── Phase 170-172: Multi-angle VTO execution ──
async def process_vto_synthesis(job_id: str, image_bytes: bytes, user_id: str):
    """
    Full multi-angle VTO synthesis pipeline:
    1. SAM2 persona masking (Phase 157)
    2. Side-to-front alignment (Phase 158)
    3. Neutralization (Phase 159)
    4. UV projection (Phase 160)
    5. Back-view synthesis with VLM in-paint (Phase 161-162)
    6. Super-resolution (Phase 164)
    7. Symmetry verification (Phase 165)
    """
    try:
        _vto_emit(job_id, "loading", 5, "Loading image...")
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

        # Phase 157: SAM2 persona masking
        _vto_emit(job_id, "segmenting", 15, "Extracting persona...")
        persona = _segment_persona(image)

        # Phase 158: Side-to-front alignment
        _vto_emit(job_id, "aligning", 25, "Aligning views...")
        side_aligned = _align_side_to_front(persona, persona)

        # Phase 159: Neutralize clothing
        _vto_emit(job_id, "neutralizing", 35, "Neutralizing clothing...")
        front_neutral = _neutralize_clothing(persona)
        side_neutral = _neutralize_clothing(side_aligned)

        # Phase 160: UV-space projection
        _vto_emit(job_id, "projecting", 45, "Projecting to UV space...")
        uv_data = _project_to_uv(front_neutral, side_neutral)

        # Phase 161-162: Back-view synthesis with VLM in-paint
        _vto_emit(job_id, "synthesizing", 55, "Synthesizing back view...")
        back_view = _synthesize_back_view(front_neutral, side_neutral, use_vlm=True)

        # Phase 164: Super-resolution enhancement
        _vto_emit(job_id, "upscaling", 70, "Enhancing resolution...")
        import cv2
        for view_name, view_img in [("front", front_neutral), ("side", side_neutral), ("back", back_view)]:
            enhanced = _enhance_detail(view_img)
            tmp_path = f"/kaggle/working/vto_{job_id}_{view_name}.png"
            enhanced.save(tmp_path, "PNG")

        # Phase 165: Symmetry verification
        _vto_emit(job_id, "verifying", 85, "Verifying symmetry...")
        sym = _verify_symmetry(front_neutral, back_view)
        logger.info(f"VTO {job_id} symmetry: {sym}")
        if not sym["pass"]:
            logger.warning(f"VTO {job_id}: symmetry check failed (diff {sym['diff_pct']}%)")

        # Store results
        vto_jobs[job_id]["status"] = "completed"
        vto_jobs[job_id]["angles"] = {
            "front": f"/kaggle/working/vto_{job_id}_front.png",
            "side": f"/kaggle/working/vto_{job_id}_side.png",
            "back": f"/kaggle/working/vto_{job_id}_back.png",
        }
        vto_jobs[job_id]["symmetry"] = sym
        vto_jobs[job_id]["uv_data"] = uv_data
        _vto_emit(job_id, "complete", 100, "VTO synthesis complete!")
        logger.info(f"VTO job {job_id} completed")

    except Exception as e:
        import traceback as tb
        vto_jobs[job_id]["status"] = "failed"
        vto_jobs[job_id]["error"] = str(e)
        _vto_emit(job_id, "error", -1, str(e)[:200])
        logger.error(f"VTO job {job_id} failed: {e}")
        try:
            (WORKING_DIR / "last_error.txt").write_text(tb.format_exc())
        except Exception:
            pass


async def process_vto_tryon(job_id: str, image_bytes: bytes, angle: str, user_id: str, seed: int = 42, garment_type: str = "casual", garment_color: str = "#000000"):
    """
    Real garment transfer pipeline:
    1. Source image → SAM2 segment → GarmentRec 3D mesh → extract garment texture
    2. Target person → detect body pose (MediaPipe)
    3. Warp garment onto target body using pose alignment
    4. Alpha blend with feathered edges
    """
    t_start = time.time()
    try:
        import random
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        _vto_emit(job_id, "loading", 5, f"Loading {angle} view...")
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        logger.info(f"VTO {job_id} ({angle}): garment_type={garment_type}, garment_color={garment_color}")

        # ── Step 1: Extract garment from source photo ──
        _vto_emit(job_id, "extracting", 15, f"Extracting garment from source...")
        garment_class = "upper" if garment_type in ("tshirt", "shirt", "blouse", "jacket", "hoodie") else "lower" if garment_type in ("pants", "shorts", "skirt") else "auto"
        extraction = _extract_garment_from_source(image, garment_class)
        garment_crop = extraction["garment_crop"]
        garment_mask = extraction["mask_crop"]
        garment_class = extraction["garment_class"]
        logger.info(f"VTO {job_id}: extracted {garment_class} garment, bbox={extraction['garment_bbox']}")

        # ── Step 2: Extract garment color for tinting ──
        _vto_emit(job_id, "analyzing", 25, f"Analyzing garment color...")
        if garment_color and garment_color != "#000000":
            garment_crop = _apply_garment_tint(garment_crop, garment_color)
            logger.info(f"VTO {job_id}: applied color tint {garment_color}")

        # ── Step 3: Detect source body pose (from source garment image) ──
        _vto_emit(job_id, "detecting", 35, f"Detecting source body pose...")
        source_pose = _detect_body_pose(image)
        logger.info(f"VTO {job_id}: source pose via {source_pose['source']}")

        # ── Step 4: Detect target body pose (same image for single-photo VTO) ──
        _vto_emit(job_id, "detecting", 45, f"Detecting target body pose...")
        target_pose = _detect_body_pose(image)  # for single-photo try-on, source=target
        logger.info(f"VTO {job_id}: target pose via {target_pose['source']}")

        # ── Step 5: Warp garment onto target body ──
        _vto_emit(job_id, "warping", 60, f"Warping {garment_class} garment onto {angle} body...")
        result = _warp_and_blend_garment(
            source_garment=garment_crop,
            source_mask=garment_mask,
            target_image=image,
            target_pose=target_pose,
            source_pose=source_pose,
            garment_color_hex=garment_color,
        )

        # ── Step 6: Generate GarmentGPT pattern if available ──
        pattern_data = None
        if garmentgpt_predictor is not None:
            _vto_emit(job_id, "patterning", 80, f"Generating sewing pattern...")
            try:
                pattern_data = run_garmentgpt(garmentgpt_predictor, extraction["garment_img"])
                logger.info(f"VTO {job_id}: GarmentGPT pattern generated")
            except Exception as e:
                logger.warning(f"VTO {job_id}: GarmentGPT pattern failed: {e}")

        # ── Step 7: Save result ──
        _vto_emit(job_id, "saving", 90, f"Saving {angle} result...")
        tmp_path = f"/kaggle/working/vto_{job_id}_{angle}.png"
        result.save(tmp_path, "PNG")

        # Save garment mesh if available
        mesh_path = None
        if extraction["mesh_data"]:
            mesh_path = f"/kaggle/working/vto_{job_id}_{angle}_mesh.npz"
            np.savez_compressed(mesh_path,
                upper_vertices=extraction["mesh_data"].get("upper", {}).get("vertices_np", np.array([])),
                upper_faces=extraction["mesh_data"].get("upper", {}).get("faces_np", np.array([])),
                lower_vertices=extraction["mesh_data"].get("lower", {}).get("vertices_np", np.array([])),
                lower_faces=extraction["mesh_data"].get("lower", {}).get("faces_np", np.array([])),
            )

        vto_jobs[job_id]["status"] = "completed"
        vto_jobs[job_id]["result_url"] = tmp_path
        vto_jobs[job_id]["mesh_url"] = mesh_path
        vto_jobs[job_id]["pattern_data"] = pattern_data
        vto_jobs[job_id]["garment_class"] = garment_class
        _vto_emit(job_id, "complete", 100, f"{angle} try-on complete!")

        # Phase 209: Track GPU cost
        elapsed = time.time() - t_start
        if user_id not in gpu_usage_log:
            gpu_usage_log[user_id] = {"total_gpu_sec": 0.0, "jobs": 0, "last_job": ""}
        gpu_usage_log[user_id]["total_gpu_sec"] += elapsed
        gpu_usage_log[user_id]["jobs"] += 1
        gpu_usage_log[user_id]["last_job"] = job_id
        vto_jobs[job_id]["gpu_seconds"] = round(elapsed, 2)
        logger.info(f"VTO try-on {job_id} ({angle}) completed: {garment_class} garment transferred ({elapsed:.1f}s GPU)")

    except Exception as e:
        import traceback as tb
        vto_jobs[job_id]["status"] = "failed"
        vto_jobs[job_id]["error"] = str(e)
        _vto_emit(job_id, "error", -1, str(e)[:200])
        logger.error(f"VTO try-on {job_id} ({angle}) failed: {e}")
        try:
            (WORKING_DIR / "last_error.txt").write_text(tb.format_exc())
        except Exception:
            pass


# ── Phase 163: Synthesize views endpoint ──
@app.post("/api/v1/synthesize-views")
async def synthesize_views(
    file: UploadFile = File(...),
    user_id: str = "",
):
    image_bytes = await file.read()
    if len(image_bytes) > 20 * 1024 * 1024:
        raise HTTPException(400, detail=_structured_error("IMAGE_TOO_LARGE"))
    if not (file.content_type or "").startswith("image/"):
        raise HTTPException(400, detail=_structured_error("INVALID_IMAGE", detail="File must be an image"))

    job_id = str(uuid.uuid4())[:8]
    vto_jobs[job_id] = {"status": "processing", "angles": {}, "error": None, "user_id": user_id, "symmetry": None, "uv_data": None}
    asyncio.create_task(process_vto_synthesis(job_id, image_bytes, user_id))
    return {"job_id": job_id, "status": "processing"}


# ── Phase 170: Single-angle try-on endpoint ──
@app.post("/api/v1/vto/tryon")
async def vto_tryon_endpoint(
    file: UploadFile = File(...),
    angle: str = "front",
    user_id: str = "",
    seed: int = 42,
    garment_type: str = "casual",
    garment_color: str = "#000000",
):
    image_bytes = await file.read()
    if len(image_bytes) > 20 * 1024 * 1024:
        raise HTTPException(400, detail=_structured_error("IMAGE_TOO_LARGE"))
    if len(image_bytes) < 100:
        raise HTTPException(400, detail=_structured_error("EMPTY_IMAGE", detail="Image is empty or too small"))
    if not (file.content_type or "").startswith("image/"):
        raise HTTPException(400, detail=_structured_error("INVALID_IMAGE", detail="File must be an image"))

    job_id = str(uuid.uuid4())[:8]
    vto_jobs[job_id] = {"status": "processing", "result_url": None, "error": None, "user_id": user_id, "angle": angle, "garment_type": garment_type, "garment_color": garment_color}
    asyncio.create_task(process_vto_tryon(job_id, image_bytes, angle, user_id, seed=seed, garment_type=garment_type, garment_color=garment_color))
    return {"job_id": job_id, "status": "processing", "seed": seed, "garment_type": garment_type, "garment_color": garment_color}


# ── Phase 185: Multi-angle VTO (all 3 angles in one call) ──
@app.post("/api/v1/vto/multi-tryon")
async def vto_multi_tryon(
    file: UploadFile = File(...),
    user_id: str = "",
):
    """
    Phase 170: Run all 3 angles (front/side/back) in parallel via asyncio.gather.
    Phase 171: Uses consistent seed across all views.
    Phase 185: Returns partial results if some angles fail.
    """
    image_bytes = await file.read()
    if len(image_bytes) > 20 * 1024 * 1024:
        raise HTTPException(400, detail=_structured_error("IMAGE_TOO_LARGE"))
    if len(image_bytes) < 100:
        raise HTTPException(400, detail=_structured_error("EMPTY_IMAGE", detail="Image is empty or too small"))
    if not (file.content_type or "").startswith("image/"):
        raise HTTPException(400, detail=_structured_error("INVALID_IMAGE", detail="File must be an image"))

    seed = 42
    job_ids = {}
    for angle in ["front", "side", "back"]:
        jid = str(uuid.uuid4())[:8]
        job_ids[angle] = jid
        vto_jobs[jid] = {"status": "processing", "result_url": None, "error": None, "user_id": user_id, "angle": angle}
        asyncio.create_task(process_vto_tryon(jid, image_bytes, angle, user_id, seed=seed))

    return {
        "job_ids": job_ids,
        "status": "processing",
        "seed": seed,
    }


# ── Phase 173: SSE progress endpoint ──
@app.get("/api/v1/vto/progress/{job_id}")
async def vto_progress_sse(job_id: str):
    if job_id not in vto_jobs:
        raise HTTPException(404, "VTO job not found")

    async def event_generator():
        last_seq = 0
        timeout_count = 0
        max_timeout = 60
        while timeout_count < max_timeout:
            progress = vto_progress.get(job_id, {})
            current_seq = progress.get("sequence", 0)
            if current_seq > last_seq:
                last_seq = current_seq
                timeout_count = 0
                data = json.dumps({
                    "stage": progress.get("stage", "unknown"),
                    "progress": progress.get("progress", 0),
                    "message": progress.get("message", ""),
                })
                yield f"data: {data}\n\n"
                if progress.get("stage") in ("complete", "error"):
                    break
            else:
                timeout_count += 1
            evt = asyncio.Event()
            vto_progress_events.setdefault(job_id, []).add(evt)
            try:
                await asyncio.wait_for(evt.wait(), timeout=5.0)
            except asyncio.TimeoutError:
                pass
            finally:
                vto_progress_events.get(job_id, set()).discard(evt)
        vto_progress_events.pop(job_id, None)

    return StreamingResponse(
        event_generator(),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "Connection": "keep-alive", "X-Accel-Buffering": "no"},
    )


@app.get("/api/v1/vto/status/{job_id}")
async def vto_status_endpoint(job_id: str):
    job = vto_jobs.get(job_id)
    if not job:
        raise HTTPException(404, "VTO job not found")
    progress = vto_progress.get(job_id, {})
    return {
        "status": job["status"],
        "stage": progress.get("stage", "unknown"),
        "progress": progress.get("progress", 0),
        "message": progress.get("message", ""),
        "angles": job.get("angles", {}),
        "error": job.get("error"),
        "symmetry": job.get("symmetry"),
        "angle": job.get("angle"),
    }


@app.get("/api/v1/vto/result/{job_id}")
async def vto_result(job_id: str):
    """Return VTO result image file."""
    job = vto_jobs.get(job_id)
    if not job:
        raise HTTPException(404, "VTO job not found")
    if job["status"] != "completed":
        raise HTTPException(400, "VTO job not completed")

    result_url = job.get("result_url") or (job.get("angles", {}) or {}).get("front")
    if not result_url or not os.path.exists(result_url):
        raise HTTPException(404, "Result file not found")

    return StreamingResponse(
        open(result_url, "rb"),
        media_type="image/png",
        headers={
            # Phase 202: Cache immutable VTO results for 30 days
            "Cache-Control": "public, max-age=2592000, immutable",
            "ETag": f"vto-{job_id}-{angle}",
        },
    )


# ── Phase 209: GPU cost tracking ──
@app.get("/api/v1/cost/usage")
async def gpu_cost_usage(user_id: str = ""):
    """Return GPU usage stats. If user_id provided, returns per-user stats."""
    if user_id and user_id in gpu_usage_log:
        entry = gpu_usage_log[user_id]
        return {
            "user_id": user_id,
            "total_gpu_seconds": round(entry["total_gpu_sec"], 2),
            "total_gpu_minutes": round(entry["total_gpu_sec"] / 60, 2),
            "estimated_cost_usd": round(entry["total_gpu_sec"] / 60 * COST_PER_GPU_MINUTE_USD, 4),
            "total_jobs": entry["jobs"],
            "last_job": entry["last_job"],
        }
    # Global stats
    total_sec = sum(e["total_gpu_sec"] for e in gpu_usage_log.values())
    total_jobs = sum(e["jobs"] for e in gpu_usage_log.values())
    return {
        "total_gpu_seconds": round(total_sec, 2),
        "total_gpu_minutes": round(total_sec / 60, 2),
        "estimated_cost_usd": round(total_sec / 60 * COST_PER_GPU_MINUTE_USD, 4),
        "total_jobs": total_jobs,
        "unique_users": len(gpu_usage_log),
        "uptime_hours": round((time.time() - SERVER_START_TIME) / 3600, 2),
    }


# ── Phase 212: Quality dashboard ──
@app.get("/api/v1/quality/dashboard")
async def quality_dashboard():
    """
    Phase 212: Real-time quality metrics dashboard.
    Returns aggregated stats about VTO job success rates, latency, and errors.
    """
    now = time.time()
    uptime = now - SERVER_START_TIME

    # Aggregate VTO job stats
    total_vto = len(vto_jobs)
    completed = sum(1 for j in vto_jobs.values() if j.get("status") == "completed")
    failed = sum(1 for j in vto_jobs.values() if j.get("status") == "failed")
    processing = sum(1 for j in vto_jobs.values() if j.get("status") == "processing")

    # GPU memory
    gpu_info = {}
    if torch.cuda.is_available():
        gpu_info = {
            "allocated_gb": round(torch.cuda.memory_allocated() / (1024**3), 2),
            "reserved_gb": round(torch.cuda.memory_reserved() / (1024**3), 2),
            "max_allocated_gb": round(torch.cuda.max_memory_allocated() / (1024**3), 2),
        }

    # Cost tracking
    total_gpu_sec = sum(e["total_gpu_sec"] for e in gpu_usage_log.values())
    estimated_cost = round(total_gpu_sec / 60 * COST_PER_GPU_MINUTE_USD, 4)

    # Garment class distribution
    garment_dist = {}
    for j in vto_jobs.values():
        gc = j.get("garment_class", "unknown")
        garment_dist[gc] = garment_dist.get(gc, 0) + 1

    return {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "uptime_hours": round(uptime / 3600, 2),
        "vto_jobs": {
            "total": total_vto,
            "completed": completed,
            "failed": failed,
            "processing": processing,
            "success_rate": round(completed / max(total_vto, 1) * 100, 1),
        },
        "garment_distribution": garment_dist,
        "gpu": gpu_info,
        "cost": {
            "total_gpu_minutes": round(total_gpu_sec / 60, 2),
            "estimated_cost_usd": estimated_cost,
            "unique_users": len(gpu_usage_log),
        },
        "models_loaded": {
            "rembg": rembg_remove is not None,
            "sam2": sam2_model is not None,
            "garmentrec": garmentrec_model is not None,
            "garmentgpt": garmentgpt_predictor is not None,
        },
        "errors": {
            "total": ERROR_COUNT,
            "rate": round(ERROR_COUNT / max(REQUEST_COUNT, 1) * 100, 2),
        },
    }


def _start_server():
    """Start uvicorn directly unless a notebook event loop is active."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        uvicorn.run(app, host="0.0.0.0", port=8000, limit_concurrency=10, backlog=10)
        return None

    def _target():
        uvicorn.run(app, host="0.0.0.0", port=8000, limit_concurrency=10, backlog=10)

    thread = threading.Thread(target=_target, daemon=True, name="garment-uvicorn")
    thread.start()
    return thread


if __name__ == "__main__":
    import signal

    # Create event loop early to avoid nest_asyncio/uvloop conflict
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    def _graceful_exit(signum, frame):
        logger.info(f"Received signal {signum}, cleaning up GPU memory...")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        logger.info("GPU memory cleaned, exiting.")
        raise SystemExit(0)

    signal.signal(signal.SIGTERM, _graceful_exit)
    signal.signal(signal.SIGINT, _graceful_exit)
    _start_server()


In [ ]:
"""
Cell 5: Start Server + Tunnel + Keep-Alive (fp16 + OOM fix + int8 shim)
Paste this entire cell into Kaggle notebook Cell 5.
"""

# ---- vLLM compatibility layer (SHIM) ----
# Required because Garment-GPT main.py imports vllm, which isn't compatible with Kaggle P100 (sm_60).
import sys, types, torch

def _activate_vllm_shim():
    import importlib
    _orig_find_spec = importlib.util.find_spec
    def _patched_find_spec(name, *args, **kwargs):
        if name == 'vllm':
            return None
        return _orig_find_spec(name, *args, **kwargs)
    importlib.util.find_spec = _patched_find_spec

    class _HF_LLM:
        def __init__(self, model, **kwargs):
            from transformers import AutoProcessor, AutoConfig
            import torch

            use_8bit = False
            gpu_label = "cpu"
            if torch.cuda.is_available():
                cap = torch.cuda.get_device_capability()
                gpu_label = f"sm_{cap[0]}{cap[1]}"
                if cap >= (7, 0):
                    use_8bit = True

            print(f'[vllm-shim] Loading model {model} on {gpu_label}...')
            self._processor = AutoProcessor.from_pretrained(model, trust_remote_code=True)
            cfg_kwargs = dict(kwargs)
            cfg_kwargs.setdefault('trust_remote_code', True)
            cfg = AutoConfig.from_pretrained(model, **cfg_kwargs)
            if hasattr(cfg, 'vision_config'):
                try:
                    from transformers import AutoModelForVision2Seq
                    model_class = AutoModelForVision2Seq
                except ImportError:
                    from transformers import LlavaForConditionalGeneration
                    model_class = LlavaForConditionalGeneration
            else:
                from transformers import AutoModelForCausalLM
                model_class = AutoModelForCausalLM

            load_kwargs = dict(
                device_map='auto', trust_remote_code=True, torch_dtype=torch.float16,
            )
            if use_8bit:
                from transformers import BitsAndBytesConfig
                load_kwargs['quantization_config'] = BitsAndBytesConfig(
                    load_in_8bit=True, llm_int8_threshold=6.0,
                )

            self._model = model_class.from_pretrained(model, **load_kwargs)
            self._model.eval()
            print(f'[vllm-shim] Model loaded on {self._model.device}')

        def generate(self, inputs_list, sampling_params):
            results = []
            for inp in inputs_list:
                prompt = inp.get('prompt', '')
                images = inp.get('multi_modal_data', {}).get('image', [])
                if images:
                    proc = self._processor(text=prompt, images=images[0], return_tensors='pt').to(self._model.device)
                else:
                    proc = self._processor(text=prompt, return_tensors='pt').to(self._model.device)
                mt = getattr(sampling_params, 'max_tokens', 4096)
                tp = getattr(sampling_params, 'temperature', 0.1)
                with torch.no_grad():
                    out = self._model.generate(**proc, max_new_tokens=min(mt, 4096),
                        do_sample=tp > 0, temperature=max(tp, 0.01), top_p=0.9)
                gen = self._processor.decode(out[0], skip_special_tokens=False)
                class _O:
                    def __init__(self, text): self.text = text
                class _RO:
                    def __init__(self, outputs): self.outputs = outputs
                results.append(_RO(outputs=[_O(text=gen)]))
            return results

    class _HF_SamplingParams:
        def __init__(self, temperature=0.1, max_tokens=4096, skip_special_tokens=False, seed=42, stop_token_ids=None):
            self.temperature = temperature
            self.max_tokens = max_tokens
            self.skip_special_tokens = skip_special_tokens
            self.seed = seed
            self.stop_token_ids = stop_token_ids or []

    vllm_mod = types.ModuleType('vllm')
    vllm_mod.LLM = _HF_LLM
    vllm_mod.SamplingParams = _HF_SamplingParams
    sys.modules['vllm'] = vllm_mod
    print('[vllm-shim] vLLM compatibility shim installed (transformers + int8 backend)')

_activate_vllm_shim()

import os, subprocess, time, re, requests, shutil, threading
def register_tunnel(url):
    """POST tunnel URL to EC2 proxy + write output file for rotation script."""
    if not url or not url.startswith("http"):
        print(f"[Tunnel] Skipping registration: invalid URL '{url}'")
        return
    from pathlib import Path
    out = Path("/root/output")
    out.mkdir(exist_ok=True)
    out.joinpath("tunnel_url.txt").write_text(url)
    print(f"[Tunnel] Written to {out/'tunnel_url.txt'}")
    for attempt in range(5):
        try:
            import requests as _req
            r = _req.post("https://korra.work/api/v2/garment/internal/tunnel",
                          json={"url": url}, timeout=10)
            body = r.text[:500]
            if r.status_code == 200:
                print(f"[Tunnel] Registered with EC2 proxy (attempt {attempt+1}): {body[:200]}")
                return
            print(f"[Tunnel] EC2 proxy HTTP {r.status_code} (attempt {attempt+1}): {body[:200]}")
        except Exception as e:
            print(f"[Tunnel] EC2 proxy error (attempt {attempt+1}): {e}")
        time.sleep(5)


# Locate cloudflared binary
CLOUDFLARED = shutil.which("cloudflared") or "/root/cloudflared"
if not os.path.exists(CLOUDFLARED):
    print(f"cloudflared not found at {CLOUDFLARED}, downloading...")
    import urllib.request
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CLOUDFLARED)
    os.chmod(CLOUDFLARED, 0o755) if os.path.exists(CLOUDFLARED) else None
    print(f"cloudflared downloaded: {os.path.getsize(CLOUDFLARED) / 1024:.0f} KB")

# Kill any old processes
os.system("pkill -f api_server.py 2>/dev/null || true")
os.system("pkill -f cloudflared 2>/dev/null || true")
# Kill anything on port 8000
os.system(r"kill -9 $(ss -tlnp sport = :8000 2>/dev/null | grep -oP 'pid=\K[0-9]+' 2>/dev/null) 2>/dev/null || true")
os.system("for p in $(ls /proc/*/fd 2>/dev/null | grep -l socket 2>/dev/null | head -20); do kill -9 $(cat $p/../status 2>/dev/null | grep NSpid | awk '{print $2}') 2>/dev/null; done || true")
time.sleep(3)

tunnel_url = None
_tunnel_start_time = None
_tunnel_restarts = 0
_server_restarts = 0
_keep_alive_backoff = 60  # initial check interval in seconds
_max_server_restarts = 10  # after this, enter degraded mode
_degraded_mode = False
_watchdog_restarts = 0
_max_watchdog_restarts = 5

def _tunnel_reader(stream):
    """Read tunnel output, print it, and watch for tunnel URL."""
    global tunnel_url, _tunnel_start_time, _tunnel_restarts
    try:
        for line in iter(stream.readline, ''):
            if line:
                line = line.rstrip("\n")
                print(f"[TUN] {line}")
                if not tunnel_url:
                    m = re.search(r'https?://[^\s]*trycloudflare\.com[^\s]*', line)
                    if m:
                        tunnel_url = m.group(0)
                        _tunnel_start_time = time.time()
                        _tunnel_restarts += 1
                        print(f"\n{'='*60}\nTUNNEL URL: {tunnel_url}\n{'='*60}")
    except ValueError:
        pass
    finally:
        try:
            stream.close()
        except Exception:
            pass


def start_tunnel():
    global tunnel_url
    # Reset stale tunnel URL so _tunnel_reader captures fresh one
    tunnel_url = None
    print(f"[TUN] Starting cloudflared from {CLOUDFLARED}...")
    p = subprocess.Popen(
        [CLOUDFLARED, "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    # Drain tunnel logs in background thread (prevents pipe buffer blocking)
    t = threading.Thread(target=_tunnel_reader, args=(p.stdout,), daemon=True)
    t.start()
    # Wait for tunnel URL (up to 120s)
    deadline = time.time() + 120
    while time.time() < deadline:
        if tunnel_url:
            break
        if p.poll() is not None:
            print(f"[TUN] cloudflared died (code {p.returncode}), restarting...")
            return start_tunnel()
        time.sleep(2)
    if not tunnel_url:
        print(f"[TUN] WARNING: tunnel URL not obtained within 120s, continuing in background")
    else:
        print(f"[TUN] Tunnel URL obtained: {tunnel_url}")
    return p


def _pipe_drainer(stream, prefix, log_file=None):
    """Read from a subprocess pipe continuously and print to stdout."""
    try:
        for line in iter(stream.readline, ''):
            if line:
                print(f"{prefix} {line.strip()}")
                if log_file:
                    log_file.write(line)
                    log_file.flush()
    except ValueError:
        pass  # closed stream
    finally:
        stream.close()
        if log_file:
            log_file.close()


os.environ["GARMENT_WORKING_DIR"] = "/root"

def start_server():
    working_dir = os.getenv("GARMENT_WORKING_DIR") or "/root"
    os.makedirs(working_dir, exist_ok=True)
    api_path = os.path.join(working_dir, "api_server.py")
    if not os.path.exists(api_path):
        print(f"[ERROR] {api_path} not found! Run Cell 8 (%%writefile) first.")
        return None
    log_file = open(os.path.join(working_dir, "server_logs.txt"), "a", buffering=1)
    p = subprocess.Popen(
        [sys.executable, os.path.join(working_dir, "api_server.py")],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        env={**os.environ, "GARMENT_WORKING_DIR": working_dir},
    )
    t = threading.Thread(target=_pipe_drainer, args=(p.stdout, "[API]", log_file), daemon=True)
    t.start()
    # Wait for server to be ready (up to 60s)
    deadline = time.time() + 60
    while time.time() < deadline:
        try:
            r = requests.get("http://localhost:8000/health", timeout=3)
            if r.status_code < 500:
                print("[API] Server ready")
                break_on_fail = False
                break
        except Exception:
            time.sleep(1)
            continue
    return p


def kill_server(proc):
    """Gracefully kill server: SIGTERM -> wait 5s -> SIGKILL. Frees CUDA memory."""
    if proc is None:
        return
    pid = proc.pid
    if pid is None:
        return
    try:
        os.kill(pid, 0)  # Check if process exists
    except OSError:
        return  # Already dead
    print(f"[KEEP] Sending SIGTERM to server (PID {pid})...")
    try:
        proc.terminate()  # SIGTERM
        proc.wait(timeout=8)
        print(f"[KEEP] Server PID {pid} terminated gracefully")
    except subprocess.TimeoutExpired:
        print(f"[KEEP] Server PID {pid} didn't exit in 8s, sending SIGKILL...")
        proc.kill()  # SIGKILL
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            print(f"[KEEP] Server PID {pid} still alive after SIGKILL, moving on")
    except Exception as e:
        print(f"[KEEP] Error killing server PID {pid}: {e}")
        try:
            proc.kill()
        except Exception:
            pass
    # Force CUDA memory cleanup after kill
    import gc
    gc.collect()
    # Wait for port to be freed
    time.sleep(2)

# Start tunnel first (blocking until URL), then server
t_proc = start_tunnel()
s_proc = start_server()
_server_start_time = time.time()

# Wait then health check
time.sleep(3)
try:
    r = requests.get("http://localhost:8000/health", timeout=5)
    print(f"Health: {r.json()}")
except Exception as e:
    print(f"Health check: {e}")

# Wait up to 90s for tunnel URL
for _wait in range(30):
    if tunnel_url:
        break
    time.sleep(3)
    print(f"[Tunnel] Waiting for URL... ({_wait*3+3}s)")
    if t_proc.poll() is not None:
        print(f"[Tunnel] cloudflared died (code {t_proc.returncode}), restarting...")
        t_proc = start_tunnel()

if tunnel_url:
    print(f"\nReady! Tunnel URL: {tunnel_url}")
    register_tunnel(tunnel_url)
else:
    print(f"\nTunnel URL not yet available — will retry in keep-alive loop")

# === STARTUP TIMEOUT (Phase 44) ===
_STARTUP_TIMEOUT = 120  # max seconds to wait for server + tunnel
_startup_deadline = time.time() + _STARTUP_TIMEOUT
while time.time() < _startup_deadline:
    # Check server health
    try:
        r = requests.get("http://localhost:8000/health", timeout=5)
        j = r.json()
        server_up = True
        print(f"Health: {j}")
    except Exception as e:
        server_up = False
        print(f"Startup health check: {e}")

    if tunnel_url and server_up:
        print(f"\nReady! Tunnel URL: {tunnel_url}")
        register_tunnel(tunnel_url)
        break

    # Check tunnel
    if t_proc.poll() is not None:
        print(f"[Tunnel] cloudflared died (code {t_proc.returncode}), restarting...")
        t_proc = start_tunnel()

    # Check server
    if s_proc.poll() is not None:
        print(f"[Server] api_server died (code {s_proc.returncode}), restarting...")
        s_proc = start_server()
        _server_start_time = time.time()

    time.sleep(5)

if not tunnel_url:
    print(f"\nTunnel URL not yet available — will retry in keep-alive loop")
if not server_up:
    print(f"\nServer not responding — will retry in keep-alive loop")

# === KEEP-ALIVE LOOP (Phases 41-45) ===
print(f"\nKeep-alive running. Initial interval: {_keep_alive_backoff}s.")
import datetime as dt
_health_fails = 0
_tunnel_fails = 0

# === PHASE 45: WATCHDOG-MANAGED KEEP-ALIVE LOOP ===
def _keep_alive_loop():
    """Core keep-alive loop. Runs forever unless an exception escapes."""
    global _health_fails, _tunnel_fails, _server_restarts, _degraded_mode
    global _keep_alive_backoff, t_proc, s_proc, tunnel_url, _tunnel_start_time
    _health_fails = 0
    _tunnel_fails = 0
    while True:
        try:
            # Local health check
            r = requests.get("http://localhost:8000/health", timeout=10)
            j = r.json()
            _health_fails = 0
            # Phase 41: reset backoff on success
            _keep_alive_backoff = max(60, _keep_alive_backoff * 0.75)

            # Update tunnel_url from health if needed
            if tunnel_url and not tunnel_url.startswith('http'):
                tunnel_url = j.get('tunnel_url')

            # Phase 43: Tunnel health check via Cloudflare
            tunnel_alive_via_cloudflare = False
            if tunnel_url:
                try:
                    hr = requests.get(f"{tunnel_url}/health", timeout=10)
                    tunnel_alive_via_cloudflare = hr.status_code == 200
                except Exception:
                    pass

            tunnel_uptime = time.time() - (_tunnel_start_time or time.time())
            now = dt.datetime.now().strftime("%H:%M:%S")
            tu_label = (tunnel_url or 'WAITING')[-55:]
            alive_flag = 'CF' if tunnel_alive_via_cloudflare else 'LOCAL'
            gpu_ok = j.get('gpu_ok', True)
            mode_flag = 'DEGRADED' if _degraded_mode else 'ACTIVE'
            print(f"[{now}] {mode_flag} | GPU: {j.get('gpu','?')} ok={gpu_ok} | Tunnel: {tu_label} [{alive_flag}] up={tunnel_uptime/60:.0f}m | tunnel_restarts={_tunnel_restarts} server_restarts={_server_restarts}")

            if tunnel_url and tunnel_alive_via_cloudflare:
                register_tunnel(tunnel_url)
                _tunnel_fails = 0
            elif tunnel_url and not tunnel_alive_via_cloudflare:
                _tunnel_fails += 1
                print(f"[KEEP] Tunnel URL set but NOT reachable via Cloudflare ({_tunnel_fails}/3). Checking cloudflared process...")
                if t_proc.poll() is not None:
                    print(f"[KEEP] cloudflared died (code {t_proc.returncode}), restarting...")
                    t_proc = start_tunnel()
                    register_tunnel(tunnel_url)
                    _tunnel_start_time = time.time()
                elif _tunnel_fails >= 3:
                    print(f"[KEEP] 3 consecutive tunnel fails — killing cloudflared for fresh tunnel")
                    t_proc.kill()
                    t_proc = start_tunnel()
                    register_tunnel(tunnel_url)
                    _tunnel_start_time = time.time()
                    _tunnel_fails = 0
            elif t_proc.poll() is not None:
                print(f"[KEEP] cloudflared died (code {t_proc.returncode}), restarting...")
                t_proc = start_tunnel()
                register_tunnel(tunnel_url)
                _tunnel_start_time = time.time()
            else:
                _tunnel_fails += 1
                if _tunnel_fails >= 6:
                    print(f"[KEEP] Tunnel still waiting after ~30 min, restarting...")
                    t_proc.kill()
                    t_proc = start_tunnel()
                    register_tunnel(tunnel_url)
                    _tunnel_start_time = time.time()
                    _tunnel_fails = 0

        except Exception as e:
            now = dt.datetime.now().strftime("%H:%M:%S")
            # Distinguish "server still loading" from "server crashed"
            err_str = str(e)
            server_uptime = time.time() - _server_start_time if '_server_start_time' in dir() else 999
            is_loading = ("Expecting value" in err_str or "RemoteDisconnected" in err_str or
                          ("Connection refused" in err_str and server_uptime < 120))
            if is_loading:
                # Server is alive but returning non-JSON (loading, restarting, etc.)
                print(f"[{now}] Health check: server still loading ({int(server_uptime)}s uptime): {err_str[:80]}")
                _health_fails = 0  # Don't count as failure
                _keep_alive_backoff = max(60, _keep_alive_backoff * 0.75)
                time.sleep(int(_keep_alive_backoff))
                continue

            _health_fails += 1
            # Phase 41: exponential backoff on failure
            _keep_alive_backoff = min(600, _keep_alive_backoff * 2.0)

            if _degraded_mode:
                print(f"[{now}] DEGRADED MODE: health fail ({_health_fails}/3), server restart SKIPPED (max restarts {_max_server_restarts} reached)")
                continue

            if _health_fails >= 3:
                _server_restarts += 1
                # Phase 42: max restart limit
                if _server_restarts > _max_server_restarts:
                    _degraded_mode = True
                    print(f"[{now}] Entering DEGRADED MODE after {_server_restarts} server restarts. Tunnel will stay up for manual recovery.")
                    continue

                print(f"[{now}] Health fail {_health_fails}/3 — restarting server ({_server_restarts}/{_max_server_restarts})")
                kill_server(s_proc)
                s_proc = start_server()
                _server_start_time = time.time()
                _health_fails = 0
            else:
                print(f"[{now}] Health fail ({_health_fails}/3): {type(e).__name__}: {e} — not restarting yet")

        # Phase 41: adaptive sleep interval
        time.sleep(int(_keep_alive_backoff))


def _watchdog_main():
    """Watchdog wrapper: restarts _keep_alive_loop if it crashes."""
    global _watchdog_restarts, _keep_alive_backoff, _degraded_mode
    while True:
        try:
            _keep_alive_loop()
        except Exception as e:
            _watchdog_restarts += 1
            if _watchdog_restarts > _max_watchdog_restarts:
                print(f"[WATCHDOG] Max restarts ({_max_watchdog_restarts}) reached. Giving up.")
                break
            _keep_alive_backoff = min(600, _keep_alive_backoff * 2.0)
            wait = int(_keep_alive_backoff)
            print(f"[WATCHDOG] Keep-alive crashed ({type(e).__name__}: {e}). Restarting in {wait}s... (attempt {_watchdog_restarts}/{_max_watchdog_restarts})")
            time.sleep(wait)
            _degraded_mode = False
            # Kill and restart tunnel + server from scratch
            if t_proc: t_proc.kill()
            if s_proc: kill_server(s_proc)
            time.sleep(3)
            t_proc = start_tunnel()
            s_proc = start_server()
            _server_start_time = time.time()


_watchdog_main()
